# 중간고사 — 온실 실험 데이터 분석 에이전트 · Midterm — A Greenhouse-Experiment Analysis Agent

전산실험 2 · AI Agent 만들기 · **8주차** · 3시간 · 성적 40%
Computational Experiments 2 · Building AI Agents · **Week 8** · 3 hours · 40% of the grade

---

## 제출 전 확인 · Before you submit

- [ ] 이름·학번을 아래 셀에 적었다
- [ ] 다섯 질문 모두 실행한 **출력이 남아 있다**
- [ ] 각 질문의 **실행 경로(Trace)** 가 출력된다
- [ ] API 키가 코드에 하드코딩되어 있지 **않다** (있으면 −10점)
- [ ] 내 저장소 `exam/` 폴더에 push 했다

> **EN**
> - [ ] Your name and student number are filled in below
> - [ ] **The outputs of all five questions are preserved**
> - [ ] **The execution trace** is printed for each question
> - [ ] The API key is **not** hardcoded (−10 marks if it is)
> - [ ] Pushed to the `exam/` folder of your repository

## 시간 배분 권장 · Suggested pacing

| 시각 | 할 일 |
|---|---|
| ~0:20 | 데이터 훑어보기, 함정 찾기 |
| ~1:00 | **일단 돌아가는 것**을 만든다 (완벽하지 않아도 됨) |
| ~2:00 | 데이터 정리 로직 보강, 실패 처리 |
| ~2:30 | 다섯 질문 실행, 결과 정리 |
| ~2:50 | push, 링크 제출 |

**EN**

| Time | What to do |
|---|---|
| ~0:20 | Skim the data and find the traps |
| ~1:00 | Get **something that runs** — it need not be perfect |
| ~2:00 | Strengthen the cleaning logic and the failure handling |
| ~2:30 | Run all five questions and tidy the results |
| ~2:50 | Push and submit the link |

> 매년 **완벽하게 만들려다 아무것도 제출 못 하는 경우**가 나옵니다.
> 1시간 지점에서 일단 동작하는 것을 만들고 그 다음에 개선하십시오.
>
> **EN** — Every year someone **submits nothing because they were perfecting it.**
> Have something working by the one-hour mark, then improve it.

In [ ]:
# 여기에 본인 정보를 적으세요 / Fill in your details here
이름 = ""
학번 = ""

assert 이름 and 학번, "이름과 학번을 입력하세요."
print(f"{학번} {이름}")

---
## 0. 환경 설정 · Environment setup

아래 준비 셀이 공용 모듈을 파일로 만들어 줍니다. 내려받지 않습니다.
The setup cells below write the shared modules as files — nothing is downloaded.
**필요한 것만 골라 쓰세요.** 전부 쓸 필요는 없습니다.
**Use only what you need** — you are not expected to use all of them.

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

EN — Shared LLM wrapper for "Computational Experiments 2: Building AI Agents".
Calls the Keimyung Statistics department server and Google Gemini the same way.
Both expose an OpenAI-compatible API, so we use the single openai SDK and only
swap `base_url` and `model`.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기

 EN — This repository is PUBLIC, so never write the department server URL or
 the API key in code. Both are announced in class; store them in Colab Secrets.
     Key icon (🔑) on the left → new secret → turn on notebook access
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예 · Example:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
                                         # Check the server is alive first
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로 / Fallback route
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정 / Server configuration
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
# EN — The department server URL is never written here; it is read from the
#      Colab secret KMU_BASE_URL. The model tag is harmless to publish, so it
#      stays a constant.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
# EN — Fallback route. Get a free key from Google AI Studio. (public URL)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다 / read from Secrets at runtime
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기 / Loading secrets
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    EN — Look up a configuration value in a safe order:
    1) Colab Secrets (key icon on the left)  2) environment variable  3) prompt.
    The point is to keep the value out of the source. The moment a notebook or
    module is pushed to GitHub, anything written in the code is world-readable —
    the internal server URL just as much as the API key.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
             EN — True hides the typed value (getpass). Pass False for values
             like a URL where you need to see typos.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수 / environment variable
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력 / type it in
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다 / ask only once per runtime
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)

    EN — Fetch the API key. Alias of get_secret, kept under the week-1 name.
    """
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.

    EN — Return the server URL with any trailing '/' removed. The department
    URL is never stored in the repository, so it comes from Secrets. If you
    saved the full '/chat/completions' path, it is trimmed for you.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    # EN — The openai SDK appends /chat/completions itself, so strip it here in
    #      case the whole path was saved as the secret.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용 / reuse within the session
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트 / LLM client
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    EN — A thin client wrapping any OpenAI-compatible server.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
        EN — Which server to use. Defaults to the department server.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
        EN — Override the defaults. You normally leave these alone.
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        # EN — Never print the URL; screen shares and screenshots leak it easily.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 / Core calls -------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용.

        EN — Return the full response object, so you can inspect `usage`
        (token counts).
        """
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다.

        EN — Send the message list and return only the response text.
        """
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과.

        EN — Yield the response chunk by chunk, giving the typing effect.
        """
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 / Convenience methods ---------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다.

        EN — A one-shot question, for when building `messages` by hand is
        tedious.
        """
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.

        EN — Retry transient network/server errors, so that a lab with many
        calls (a 30-repetition experiment, say) does not die on one failure.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반 / any network or server error
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다 / back off progressively
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다.

        EN — Ask the server how many input tokens the messages use; the output
        is capped at 1 token to avoid waste.
        """
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수 / Module-level convenience functions
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.

    EN — The simplest one-line call. Internally it builds one LLM object and
    reuses it.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단 / Server diagnostics
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    EN — Check that the server is alive and see which models it serves. This is
    the very first lab of the course; on failure it prints what to check.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    # EN — Never print the full URL; show the host masked.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다.

    EN — Mask the host so screen shares and screenshots do not leak it.
    """
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들. / What to check when the connection fails."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
%%writefile structured.py
"""전산실험2 3주차 : 구조화 출력 — 파싱 · 검증 · 재시도.

소형 모델은 "JSON으로만 답하라"고 지시해도 자주 어긴다.
코드펜스로 감싸거나, 인사를 덧붙이거나, 따옴표를 빠뜨린다.

이 모듈은 세 겹의 방어를 제공한다.

    1) 프롬프트로 형식을 강제하고        (build_json_prompt)
    2) 지저분한 응답에서 JSON을 긁어내고  (extract_json)
    3) 스키마로 검증한 뒤, 실패하면       (validate)
       오류를 되먹여 다시 요구한다        (ask_json)

EN — Week 3: structured output — parse, validate, retry.
Small models break the "answer in JSON only" rule all the time: they wrap the
answer in a code fence, add a greeting, or drop a quote. This module gives you
three layers of defence: force the format in the prompt (build_json_prompt),
scrape the JSON out of a messy reply (extract_json), validate it against a
schema (validate) and, on failure, feed the error back and ask again (ask_json).

사용 예 · Example:
    from kmu_llm import LLM
    from structured import ask_json

    schema = {
        "required": ["test", "reason"],
        "types": {"test": str, "reason": str},
        "allowed": {"test": ["t-test", "anova", "chi-square"]},
    }
    result = ask_json("두 그룹의 평균 비교에 쓸 검정은?", schema=schema)
    print(result["test"])
"""

from __future__ import annotations

import json
import re
from typing import Any

__all__ = [
    "JSONParseError",
    "SchemaError",
    "extract_json",
    "parse_json",
    "validate",
    "build_json_prompt",
    "ask_json",
]


class JSONParseError(ValueError):
    """모델 응답에서 JSON을 뽑아내지 못했을 때.

    EN — Raised when no JSON could be extracted from the model's reply.
    """


class SchemaError(ValueError):
    """JSON은 얻었지만 요구한 구조가 아닐 때.

    EN — Raised when JSON was obtained but does not match the required shape.
    """


# --------------------------------------------------------------------------
# 1) 지저분한 응답에서 JSON 긁어내기 / Scraping JSON out of a messy reply
# --------------------------------------------------------------------------
_FENCE = re.compile(r"```(?:json|python)?\s*(.*?)```", re.DOTALL)


def extract_json(text: str) -> str:
    """모델 응답에서 JSON 문자열만 뽑아낸다.

    소형 모델이 실제로 내놓는 형태들:
        ```json\\n{...}\\n```
        네, 알겠습니다! {...}
        {...}\\n도움이 되었길 바랍니다.

    EN — Pull just the JSON string out of the model's reply. The shapes above
    are what small models actually return: a fenced block, a greeting before
    the object, or a closing pleasantry after it.

    Returns
    -------
    str : JSON 으로 보이는 부분 / the part that looks like JSON

    Raises
    ------
    JSONParseError : 중괄호/대괄호 쌍을 찾지 못한 경우
                     EN — no matching brace/bracket pair was found
    """
    if not text or not text.strip():
        raise JSONParseError("응답이 비어 있습니다.")

    # (1) 코드펜스가 있으면 그 안쪽을 우선 사용
    #     EN — if there is a code fence, prefer what is inside it
    fence = _FENCE.search(text)
    if fence:
        text = fence.group(1)

    # (2) 가장 바깥쪽 { } 또는 [ ] 를 찾는다
    #     EN — find the outermost { } or [ ] pair
    for opener, closer in (("{", "}"), ("[", "]")):
        start = text.find(opener)
        end = text.rfind(closer)
        if start != -1 and end > start:
            return text[start : end + 1].strip()

    raise JSONParseError(
        f"JSON 구조를 찾지 못했습니다. 응답 앞부분: {text[:100]!r}"
    )


def parse_json(text: str) -> Any:
    """extract_json + json.loads. 실패하면 JSONParseError.

    EN — extract_json followed by json.loads; raises JSONParseError on failure.
    """
    snippet = extract_json(text)
    try:
        return json.loads(snippet)
    except json.JSONDecodeError as exc:
        # 흔한 사고: 마지막 항목 뒤의 쉼표(trailing comma)
        # EN — the usual accident: a trailing comma after the last item
        repaired = re.sub(r",\s*([}\]])", r"\1", snippet)
        try:
            return json.loads(repaired)
        except json.JSONDecodeError:
            pass
        raise JSONParseError(
            f"JSON 파싱 실패 ({exc.msg}, {exc.lineno}행 {exc.colno}열). "
            f"문제 부분: {snippet[max(0, exc.pos - 30) : exc.pos + 30]!r}"
        ) from exc


# --------------------------------------------------------------------------
# 2) 스키마 검증 / Schema validation
# --------------------------------------------------------------------------
def validate(data: Any, schema: dict | None) -> list[str]:
    """아주 단순한 스키마 검증기. 문제점 목록을 돌려준다 (빈 목록 = 통과).

    스키마 형식::

        {
            "required": ["test", "reason"],          # 반드시 있어야 할 키
            "types":    {"alpha": float, "n": int},  # 값의 파이썬 타입
            "allowed":  {"test": ["t-test", ...]},   # 허용되는 값 목록
            "range":    {"alpha": (0.0, 1.0)},       # 수치 범위 (닫힌 구간)
        }

    pydantic 같은 라이브러리를 쓸 수도 있지만, 여기서는 검증이 무엇을 하는
    일인지 보이도록 직접 구현한다.

    EN — A deliberately tiny schema validator. It returns a list of problems;
    an empty list means the data passed. The schema keys above are: required
    keys, expected Python types, allowed value lists, and closed numeric
    ranges. A library such as pydantic would do this for us, but we write it by
    hand so you can see what validation actually is.
    """
    if schema is None:
        return []

    problems: list[str] = []

    if not isinstance(data, dict):
        return [f"최상위가 객체(dict)가 아니라 {type(data).__name__} 입니다."]

    for key in schema.get("required", []):
        if key not in data:
            problems.append(f"필수 키 '{key}' 가 없습니다.")

    for key, expected in schema.get("types", {}).items():
        if key not in data:
            continue
        value = data[key]
        # int 는 float 자리에 들어와도 허용한다 / an int is acceptable where a float is expected
        if expected is float and isinstance(value, int) and not isinstance(value, bool):
            continue
        if not isinstance(value, expected):
            problems.append(
                f"'{key}' 는 {expected.__name__} 여야 하는데 "
                f"{type(value).__name__} 입니다 (값: {value!r})."
            )

    for key, options in schema.get("allowed", {}).items():
        if key in data and data[key] not in options:
            problems.append(
                f"'{key}' 의 값 {data[key]!r} 는 허용 목록에 없습니다. "
                f"가능한 값: {options}"
            )

    for key, (low, high) in schema.get("range", {}).items():
        if key not in data:
            continue
        value = data[key]
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            problems.append(f"'{key}' 가 숫자가 아니어서 범위를 확인할 수 없습니다.")
        elif not (low <= value <= high):
            problems.append(f"'{key}' 의 값 {value} 가 [{low}, {high}] 범위 밖입니다.")

    return problems


# --------------------------------------------------------------------------
# 3) 프롬프트로 형식 강제하기 / Forcing the format through the prompt
# --------------------------------------------------------------------------
def build_json_prompt(prompt: str, schema: dict | None = None, example: Any = None) -> str:
    """JSON 출력을 유도하는 지시문을 프롬프트에 덧붙인다.

    소형 모델에서 실제로 효과가 있었던 순서대로 넣는다.
    1) 출력 예시  2) 금지 사항 명시  3) 시작·끝 문자 지정

    EN — Append instructions that steer the model toward JSON output, in the
    order that actually worked on small models: (1) show an example, (2) state
    what is forbidden, (3) pin down the first and last character.
    """
    parts = [prompt, "", "## 출력 형식"]

    if example is not None:
        parts.append("아래와 정확히 같은 형식의 JSON으로만 답하세요.")
        parts.append(json.dumps(example, ensure_ascii=False, indent=2))
    elif schema and schema.get("required"):
        keys = ", ".join(f'"{k}"' for k in schema["required"])
        parts.append(f"다음 키를 가진 JSON 객체로만 답하세요: {keys}")

    if schema:
        for key, options in schema.get("allowed", {}).items():
            parts.append(f'- "{key}" 는 다음 중 하나여야 합니다: {options}')

    parts += [
        "",
        "규칙:",
        "- 설명, 인사, 사족을 쓰지 마세요.",
        "- 코드블록 표시(백틱)를 쓰지 마세요.",
        "- 응답은 반드시 { 로 시작해서 } 로 끝나야 합니다.",
    ]
    return "\n".join(parts)


# --------------------------------------------------------------------------
# 4) 파싱 · 검증 · 재시도를 합친 것 / Parse + validate + retry, combined
# --------------------------------------------------------------------------
def ask_json(
    prompt: str,
    system: str | None = None,
    *,
    schema: dict | None = None,
    example: Any = None,
    n_retry: int = 3,
    llm: Any = None,
    verbose: bool = False,
    **kw: Any,
) -> Any:
    """모델에게 JSON을 받아 파싱·검증한 결과를 돌려준다.

    실패하면 **오류 메시지를 모델에게 되돌려주고 다시 요구한다.**
    이 되먹임 구조가 6주차 ReAct 루프의 핵심 아이디어와 같다.

    EN — Get JSON from the model and return the parsed, validated result. On
    failure the error message is fed back to the model and the request is
    repeated. That feedback loop is the same core idea as the week-6 ReAct loop.

    Parameters
    ----------
    schema  : validate() 형식의 스키마 / a schema in validate()'s format
    example : 출력 예시 (few-shot). 소형 모델에서 가장 효과가 크다
              EN — an output example (few-shot); the biggest win on small models
    n_retry : 최대 시도 횟수 / maximum number of attempts
    llm     : LLM 인스턴스. 없으면 kmu_llm.LLM() 을 만든다
              EN — an LLM instance; one is created if omitted
    verbose : 시도 과정을 출력 / print each attempt

    Raises
    ------
    JSONParseError / SchemaError : n_retry 회 모두 실패한 경우
                                   EN — when all n_retry attempts failed
    """
    if llm is None:
        from kmu_llm import LLM

        llm = LLM()

    kw.setdefault("temperature", 0.0)  # 형식을 지켜야 하므로 낮게 / low, because format matters

    messages: list[dict[str, str]] = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": build_json_prompt(prompt, schema, example)})

    last_error: Exception | None = None

    for attempt in range(1, n_retry + 1):
        raw = llm.chat(messages, **kw)
        if verbose:
            print(f"[시도 {attempt}] 원문: {raw[:120]!r}")

        try:
            data = parse_json(raw)
        except JSONParseError as exc:
            last_error = exc
            feedback = (
                f"방금 응답을 JSON으로 읽을 수 없었습니다. 오류: {exc}\n"
                "설명 없이 JSON 객체만 다시 출력하세요."
            )
        else:
            problems = validate(data, schema)
            if not problems:
                if verbose:
                    print(f"[시도 {attempt}] 성공")
                return data
            last_error = SchemaError("; ".join(problems))
            feedback = (
                "JSON 형식은 맞지만 내용에 문제가 있습니다:\n"
                + "\n".join(f"- {p}" for p in problems)
                + "\n이 문제들을 고쳐서 JSON 객체만 다시 출력하세요."
            )

        if verbose:
            print(f"[시도 {attempt}] 실패 -> 되먹임: {last_error}")

        # 실패한 응답과 오류를 대화에 남겨서 모델이 스스로 고치게 한다
        # EN — keep the failed reply and the error in the conversation so the
        #      model can correct itself
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": feedback})

    raise type(last_error)(  # type: ignore[misc]
        f"{n_retry}회 시도 모두 실패했습니다. 마지막 오류: {last_error}"
    )


In [ ]:
%%writefile memory.py
"""전산실험2 4주차 : 대화 상태와 메모리.

모델은 이전 대화를 기억하지 못한다. 기억은 **우리 코드가** 만든다.
그런데 대화를 통째로 다시 보내다 보면 컨텍스트 윈도우에 부딪힌다.

이 모듈은 세 가지를 제공한다.

    Conversation   대화 기록을 들고 있는 그릇 (system 은 항상 보존)
    예산 관리       토큰을 세고, 넘치면 오래된 턴을 잘라낸다
    요약 압축       잘라내는 대신 요약으로 갈아끼운다 (정보 손실 최소화)

EN — Week 4: conversation state and memory.
The model remembers nothing; OUR code creates the memory. But resending the
whole conversation eventually hits the context window. This module offers:
Conversation (a container for the history, with the system message always
preserved), budget management (count tokens and drop old turns when over), and
summary compression (replace old turns with a summary instead of dropping them).

사용 예 · Example:
    from kmu_llm import LLM
    from memory import Conversation

    llm = LLM()
    conv = Conversation(llm, system="너는 통계 조교다.", max_tokens=2000)
    print(conv.say("표준편차가 뭔가요?"))
    print(conv.say("그럼 표준오차는요?"))     # 앞 대화를 기억한다 / remembers
    conv.summary()                          # 현재 토큰 사용 현황 / token usage
"""

from __future__ import annotations

from typing import Any, Literal

__all__ = ["estimate_tokens", "Message", "Conversation"]

Role = Literal["system", "user", "assistant"]


# --------------------------------------------------------------------------
# 토큰 추정 / Token estimation
# --------------------------------------------------------------------------
def estimate_tokens(text: str) -> int:
    """토큰 수를 대략 추정한다. 서버에 묻지 않고 빠르게 계산한다.

    정확한 값은 모델의 토크나이저에 따라 다르다. 여기서는 경험적 근사를 쓴다.

        한글 1글자  ~ 1.2 토큰   (자소 단위로 잘게 쪼개짐)
        영문 1글자  ~ 0.3 토큰   (단어가 통째로 1토큰이 되는 경우가 많음)

    루프를 돌 때마다 서버에 물어보면 호출이 배로 늘어난다. 예산 관리는
    **근사로 충분**하고, 정확한 값이 필요하면 `LLM.count_tokens()` 를 쓴다.

    EN — Roughly estimate the token count locally, without asking the server.
    The exact number depends on the model's tokenizer; these are empirical
    approximations (one Hangul character ≈ 1.2 tokens because it splits into
    jamo; one Latin character ≈ 0.3 tokens because whole words often become one
    token). Asking the server on every loop iteration would double the number of
    calls — an approximation is good enough for budgeting, and `LLM.count_tokens()`
    is there when you need the exact figure.
    """
    if not text:
        return 0
    hangul = sum(1 for ch in text if "가" <= ch <= "힣")
    other = len(text) - hangul
    return int(hangul * 1.2 + other * 0.3) + 1


class Message(dict):
    """{"role": ..., "content": ...} 에 토큰 수를 얹은 것.

    EN — A {"role": ..., "content": ...} dict with a token count attached.
    """

    def __init__(self, role: Role, content: str) -> None:
        super().__init__(role=role, content=content)

    @property
    def tokens(self) -> int:
        return estimate_tokens(self["content"]) + 4  # 역할 표시 등 부대 비용 / overhead for the role marker

    def as_api(self) -> dict[str, str]:
        return {"role": self["role"], "content": self["content"]}


# --------------------------------------------------------------------------
# 대화 / Conversation
# --------------------------------------------------------------------------
class Conversation:
    """대화 기록을 관리하고 예산을 넘지 않게 유지한다.

    EN — Keeps the conversation history and holds it within the token budget.

    Parameters
    ----------
    llm : LLM 인스턴스 / an LLM instance
    system : 대화 전체에 적용되는 지시. **절대 잘려나가지 않는다.**
             EN — the instruction for the whole conversation; never truncated.
    max_tokens : 전체 대화의 토큰 예산 (응답 공간은 별도)
                 EN — token budget for the conversation (reply space is separate)
    strategy :
        "drop"    오래된 턴을 그냥 버린다 (빠르고 단순, 정보 손실)
                  EN — discard old turns (fast and simple, loses information)
        "summary" 오래된 턴을 요약으로 갈아끼운다 (호출 1회 추가, 손실 적음)
                  EN — replace old turns with a summary (one extra call, less loss)
        "none"    아무것도 하지 않는다 (한계를 직접 보고 싶을 때)
                  EN — do nothing, so you can watch the limit being hit
    keep_recent : 최근 몇 개의 메시지는 무조건 원본으로 보존할지
                  EN — how many recent messages are always kept verbatim
    """

    def __init__(
        self,
        llm: Any,
        system: str | None = None,
        *,
        max_tokens: int = 3000,
        strategy: str = "summary",
        keep_recent: int = 4,
    ) -> None:
        if strategy not in ("drop", "summary", "none"):
            raise ValueError(f"알 수 없는 strategy: {strategy!r}")

        self.llm = llm
        self.max_tokens = max_tokens
        self.strategy = strategy
        self.keep_recent = keep_recent

        self.system = Message("system", system) if system else None
        self.history: list[Message] = []
        self.compressions = 0          # 몇 번 압축했는지 / how many compressions so far
        self.dropped = 0               # 몇 개를 버렸는지 / how many messages were dropped

    # -- 상태 확인 / Inspecting the state ----------------------------------
    def messages(self) -> list[dict[str, str]]:
        """API 로 보낼 형태. / The shape that goes to the API."""
        head = [self.system] if self.system else []
        return [m.as_api() for m in head + self.history]

    @property
    def tokens(self) -> int:
        head = self.system.tokens if self.system else 0
        return head + sum(m.tokens for m in self.history)

    def summary(self) -> dict[str, Any]:
        """현재 상태를 한눈에. / The current state at a glance."""
        return {
            "메시지수": len(self.history) + (1 if self.system else 0),
            "추정토큰": self.tokens,
            "예산": self.max_tokens,
            "사용률": round(self.tokens / self.max_tokens, 3),
            "압축횟수": self.compressions,
            "버린메시지": self.dropped,
        }

    # -- 대화 / Talking ----------------------------------------------------
    def add(self, role: Role, content: str) -> None:
        self.history.append(Message(role, content))

    def say(self, text: str, **kw: Any) -> str:
        """사용자 발화를 넣고 모델의 답을 받아 기록에 남긴다.

        예산은 **보내기 전과 받은 뒤 두 번** 맞춘다. 그래야
        "대화는 항상 예산 안에 있다"는 불변식이 유지되어,
        지금 상태를 보고 다음 호출이 들어갈지 예측할 수 있다.

        EN — Append the user's utterance, get the model's reply, and record it.
        The budget is enforced twice — before sending and after receiving — so
        the invariant "the conversation always fits the budget" holds, which
        lets you predict from the current state whether the next call will fit.
        """
        self.add("user", text)
        self.fit()                                  # 보내기 전 / before sending
        reply = self.llm.chat(self.messages(), **kw)
        self.add("assistant", reply)
        self.fit()                                  # 받은 뒤 / after receiving
        return reply

    # -- 예산 관리 / Budget management -------------------------------------
    def fit(self) -> None:
        """예산을 넘으면 전략에 따라 줄인다.

        EN — Shrink the history according to the strategy when over budget.
        """
        if self.strategy == "none" or self.tokens <= self.max_tokens:
            return
        if self.strategy == "drop":
            self._drop_old()
        else:
            self._summarize_old()

    def _split(self) -> tuple[list[Message], list[Message]]:
        """(압축 대상인 오래된 것, 보존할 최근 것)

        EN — (the old part to compress, the recent part to keep)
        """
        if len(self.history) <= self.keep_recent:
            return [], self.history
        return self.history[: -self.keep_recent], self.history[-self.keep_recent :]

    def _drop_old(self) -> None:
        """가장 오래된 것부터 예산에 맞을 때까지 버린다.

        `keep_recent` 는 보존하려 하지만, 그것만으로도 예산을 넘으면
        보존 약속을 깨고 더 버린다. 마지막 메시지 하나는 남긴다 —
        그것이 사용자의 방금 질문이기 때문이다.

        EN — Drop from the oldest until the history fits. `keep_recent` is
        honoured where possible, but if even those exceed the budget the promise
        is broken and more is dropped. The last message always survives — it is
        the user's current question.
        """
        old, recent = self._split()
        while old and self.tokens > self.max_tokens:
            old.pop(0)
            self.dropped += 1
            self.history = old + recent
        self.history = old + recent

        # 최근 것만 남았는데도 넘치면, 보존 약속을 깨고 더 버린다
        # EN — if even the recent messages overflow, break the promise and drop more
        while len(self.history) > 1 and self.tokens > self.max_tokens:
            self.history.pop(0)
            self.dropped += 1

        if self.tokens > self.max_tokens:
            # 마지막 메시지 하나가 예산보다 크다. 내용을 몰래 자르지는 않는다.
            # 조용히 틀리느니 시끄럽게 알리는 편이 낫다. (3주차와 같은 원칙)
            # EN — a single message exceeds the budget. We do not silently
            #      truncate it: better to fail loudly than to be quietly wrong
            #      (the same principle as week 3).
            print(
                f"[경고] 마지막 메시지 하나가 예산({self.max_tokens} 토큰)을 "
                f"넘습니다 (현재 {self.tokens}). 예산을 늘리거나 입력을 줄이세요."
            )

    def _summarize_old(self) -> None:
        """오래된 대화를 요약 한 덩어리로 갈아끼운다.

        EN — Replace the old part of the conversation with a single summary.
        """
        old, recent = self._split()
        if not old:
            self._drop_old()          # 줄일 것이 최근 것뿐이면 버리는 수밖에 없다
                                      # EN — if only recent turns remain, dropping is the only option
            return

        기록 = "\n".join(f"[{m['role']}] {m['content']}" for m in old)
        지시 = (
            "다음은 사용자와 조수의 지난 대화다. 이후 대화를 이어가는 데 필요한 "
            "사실만 5문장 이내로 요약하라. 결정된 내용, 사용자가 밝힌 정보, "
            "아직 해결되지 않은 질문을 우선 남겨라. 인사말은 버려라.\n\n"
            f"{기록}"
        )
        try:
            요약 = self.llm.ask(지시, temperature=0.0, max_tokens=400)
        except Exception:
            self._drop_old()          # 요약 호출이 실패하면 버리기로 대체
                                      # EN — if the summary call fails, fall back to dropping
            return

        self.history = [Message("user", f"[이전 대화 요약]\n{요약}")] + recent
        self.compressions += 1

        if self.tokens > self.max_tokens:   # 요약해도 넘치면 결국 버린다 / still over: drop
            self._drop_old()

    # -- 저장 / 복원 · Saving and restoring --------------------------------
    def to_dict(self) -> dict[str, Any]:
        return {
            "system": self.system["content"] if self.system else None,
            "history": [m.as_api() for m in self.history],
            "compressions": self.compressions,
            "dropped": self.dropped,
        }

    @classmethod
    def from_dict(cls, llm: Any, data: dict[str, Any], **kw: Any) -> "Conversation":
        conv = cls(llm, system=data.get("system"), **kw)
        for m in data.get("history", []):
            conv.add(m["role"], m["content"])
        conv.compressions = data.get("compressions", 0)
        conv.dropped = data.get("dropped", 0)
        return conv


In [ ]:
%%writefile tools.py
"""전산실험2 5주차 : 도구(Tool)와 도구 레지스트리.

에이전트가 바깥 세계에 손을 뻗는 방법.

핵심 원칙:
    **계산은 모델이 아니라 도구가 한다.**

    LLM은 "어떤 도구를 어떤 인자로 부를지" 만 정한다.
    평균을 구하고 p값을 계산하는 것은 numpy 와 scipy 가 한다.
    1주차에 확인했듯 LLM은 난수도 제대로 못 뽑는다. 계산을 맡기면 안 된다.

구성:
    Tool          이름 · 설명 · 인자 명세 · 실제 함수
    ToolResult    실행 결과 (성공/실패, 값, 모델에게 돌려줄 관찰 문자열)
    ToolRegistry  도구를 등록·직렬화·호출한다
    StatToolbox   DataFrame 에 묶인 통계 도구 모음

EN — Week 5: tools and the tool registry — how an agent reaches into the world.
The governing principle is that THE TOOL COMPUTES, NOT THE MODEL. The LLM only
decides which tool to call with which arguments; numpy and scipy compute the
mean and the p-value. As week 1 showed, an LLM cannot even draw random numbers
properly, so never hand it the arithmetic.

The pieces: Tool (name, description, argument spec, the real function),
ToolResult (the outcome — success/failure, value, and the observation string
returned to the model), ToolRegistry (registers, serialises and calls tools),
and StatToolbox (statistics tools bound to one DataFrame).

사용 예 · Example:
    import pandas as pd
    from tools import StatToolbox

    box = StatToolbox(pd.read_csv("sample_clinic.csv"))
    print(box.describe())                     # 프롬프트에 넣을 도구 목록
    r = box.call("compare_two_groups",
                 {"value_column": "bmi", "group_column": "group"})
    print(r.observation())
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any, Callable

__all__ = ["Tool", "ToolResult", "ToolRegistry", "StatToolbox"]


# --------------------------------------------------------------------------
# 도구 정의 / Tool definition
# --------------------------------------------------------------------------
@dataclass
class Tool:
    """도구 하나. / A single tool.

    `parameters` 형식 · format::

        {
            "column": {"type": "str", "desc": "분석할 열 이름", "required": True},
            "alpha":  {"type": "float", "desc": "유의수준", "default": 0.05},
            "method": {"type": "str", "enum": ["pearson", "spearman"]},
        }
    """

    name: str
    description: str
    parameters: dict[str, dict[str, Any]]
    func: Callable[..., Any]

    def spec_line(self) -> str:
        """프롬프트에 넣을 한 줄 명세.

        EN — The one-line specification that goes into the prompt.
        """
        args = []
        for pname, p in self.parameters.items():
            t = p.get("type", "str")
            mark = "" if p.get("required") else "?"
            enum = f"={'|'.join(map(str, p['enum']))}" if "enum" in p else ""
            args.append(f"{pname}{mark}: {t}{enum}")
        return f"- {self.name}({', '.join(args)}) — {self.description}"

    def validate(self, args: dict[str, Any]) -> list[str]:
        """인자를 검사한다. 문제 목록을 돌려준다 (빈 목록 = 통과).

        3주차 `structured.validate()` 와 같은 발상이다.
        **모델이 준 것을 믿지 않고 코드가 확인한다.**

        EN — Check the arguments and return a list of problems; an empty list
        means they passed. Same idea as `structured.validate()` in week 3: the
        code verifies rather than trusting what the model produced.
        """
        problems: list[str] = []

        for pname, p in self.parameters.items():
            if p.get("required") and pname not in args:
                problems.append(f"필수 인자 '{pname}' 가 없습니다.")

        known = set(self.parameters)
        for given in args:
            if given not in known:
                problems.append(
                    f"'{given}' 는 이 도구의 인자가 아닙니다. 가능한 인자: {sorted(known)}"
                )

        for pname, value in args.items():
            p = self.parameters.get(pname)
            if p is None:
                continue
            if "enum" in p and value not in p["enum"]:
                problems.append(
                    f"'{pname}' 의 값 {value!r} 는 허용되지 않습니다. 가능한 값: {p['enum']}"
                )
            expected = p.get("type")
            if expected == "int" and not isinstance(value, int):
                problems.append(f"'{pname}' 는 정수여야 합니다 (받은 값: {value!r}).")
            elif expected == "float" and not isinstance(value, (int, float)):
                problems.append(f"'{pname}' 는 숫자여야 합니다 (받은 값: {value!r}).")
            elif expected == "bool" and not isinstance(value, bool):
                problems.append(f"'{pname}' 는 true/false 여야 합니다 (받은 값: {value!r}).")
            elif expected == "str" and not isinstance(value, str):
                problems.append(f"'{pname}' 는 문자열이어야 합니다 (받은 값: {value!r}).")
        return problems


@dataclass
class ToolResult:
    """도구 실행 결과. / The outcome of running a tool."""

    tool: str
    ok: bool
    value: Any = None
    error: str | None = None
    hint: str | None = None      # 실패했을 때 다음에 무엇을 하면 되는지 / what to try next after a failure

    def observation(self, max_chars: int = 1200) -> str:
        """모델에게 돌려줄 문자열.

        **실패도 관찰이다.** 조용히 넘기지 않고 무엇이 왜 안 됐는지 알려준다.
        그래야 모델이 다음 행동을 고칠 수 있다 (3주차 되먹임과 같은 구조).

        EN — The string handed back to the model. A FAILURE IS ALSO AN
        OBSERVATION: rather than swallowing it, we say what went wrong and why,
        so the model can correct its next action — the same feedback structure
        as week 3.
        """
        if not self.ok:
            msg = f"[{self.tool} 실패] {self.error}"
            if self.hint:
                msg += f"\n힌트: {self.hint}"
            return msg

        if isinstance(self.value, str):
            body = self.value
        else:
            body = json.dumps(self.value, ensure_ascii=False, indent=2, default=str)
        if len(body) > max_chars:
            body = body[:max_chars] + f"\n... (생략, 전체 {len(body)}자)"
        return f"[{self.tool} 결과]\n{body}"


# --------------------------------------------------------------------------
# 레지스트리 / Registry
# --------------------------------------------------------------------------
class ToolRegistry:
    """도구를 등록하고, 프롬프트용으로 직렬화하고, 호출한다.

    EN — Registers tools, serialises them for the prompt, and calls them.
    """

    def __init__(self) -> None:
        self._tools: dict[str, Tool] = {}
        self.log: list[ToolResult] = []      # 실행 이력 (6주차 trace 의 씨앗) / call history, the seed of the week-6 trace

    # -- 등록 / Registration ----------------------------------------------
    def register(self, tool: Tool) -> Tool:
        self._tools[tool.name] = tool
        return tool

    def tool(self, name: str, description: str, parameters: dict | None = None):
        """데코레이터로 등록한다.

            @registry.tool("mean", "평균을 구한다", {"col": {"type": "str", "required": True}})
            def _(col): ...

        EN — Register a tool with a decorator.
        """
        def deco(func: Callable[..., Any]) -> Callable[..., Any]:
            self.register(Tool(name, description, parameters or {}, func))
            return func

        return deco

    # -- 조회 / Lookup -----------------------------------------------------
    def names(self) -> list[str]:
        return sorted(self._tools)

    def __contains__(self, name: object) -> bool:
        return name in self._tools

    def __getitem__(self, name: str) -> Tool:
        return self._tools[name]

    def __len__(self) -> int:
        return len(self._tools)

    def describe(self) -> str:
        """프롬프트에 그대로 넣을 도구 목록.

        EN — The tool list, ready to paste straight into the prompt.
        """
        lines = [t.spec_line() for _, t in sorted(self._tools.items())]
        return "\n".join(lines)

    # -- 실행 / Execution --------------------------------------------------
    def call(self, name: str, args: dict[str, Any] | None = None) -> ToolResult:
        """도구를 실행한다. **예외를 밖으로 던지지 않는다.**

        에이전트 루프 안에서 도구가 예외를 던지면 루프 전체가 죽는다.
        실패를 `ToolResult` 로 감싸 돌려주면 모델이 보고 고칠 수 있다.

        EN — Run a tool. IT NEVER RAISES. If a tool raised inside the agent
        loop the whole loop would die; wrapping the failure in a `ToolResult`
        lets the model see it and fix it.
        """
        args = args or {}

        if name not in self._tools:
            result = ToolResult(
                name, False,
                error=f"'{name}' 라는 도구가 없습니다.",
                hint=f"사용 가능한 도구: {self.names()}",
            )
            self.log.append(result)
            return result

        tool = self._tools[name]
        problems = tool.validate(args)
        if problems:
            result = ToolResult(
                name, False,
                error="인자가 잘못되었습니다: " + "; ".join(problems),
                hint=tool.spec_line(),
            )
            self.log.append(result)
            return result

        try:
            value = tool.func(**args)
            result = ToolResult(name, True, value=value)
        except Exception as exc:
            result = ToolResult(
                name, False,
                error=f"{type(exc).__name__}: {exc}",
                hint="인자 값을 확인하세요. 열 이름은 list_columns 로 확인할 수 있습니다.",
            )
        self.log.append(result)
        return result


# --------------------------------------------------------------------------
# 통계 도구 모음 / Statistics toolbox
# --------------------------------------------------------------------------
class StatToolbox(ToolRegistry):
    """DataFrame 하나에 묶인 통계 도구들.

    도구는 **작고, 하나만 하고, 결과를 구조화해서** 돌려줘야 한다.
    거대한 만능 도구를 하나 만들면 모델이 인자를 제대로 채우지 못한다.

    EN — Statistics tools bound to a single DataFrame. A tool should be SMALL,
    do ONE thing, and return a STRUCTURED result. Build one giant do-everything
    tool and the model will fail to fill in its arguments.
    """

    def __init__(self, df: Any, name: str = "data") -> None:
        super().__init__()
        self.df = df
        self.name = name
        self._register_all()

    # -- 내부 헬퍼 / Internal helpers --------------------------------------
    def _numeric(self, column: str):
        """열을 수치로 강제 변환하고 결측을 뺀다.

        실제 데이터에는 '58세', ' 47 ' 같은 값이 섞여 있다.
        도구가 이것을 조용히 통과시키면 이후 계산이 전부 틀어진다.

        EN — Coerce a column to numeric and drop the missing values. Real data
        contains things like '58세' or ' 47 '; if a tool lets those slip through
        silently, every calculation downstream is wrong.
        """
        import pandas as pd

        if column not in self.df.columns:
            raise KeyError(f"'{column}' 열이 없습니다. 있는 열: {list(self.df.columns)}")
        s = pd.to_numeric(self.df[column], errors="coerce")
        n_bad = int(s.isna().sum() - self.df[column].isna().sum())
        s = s.dropna()
        if len(s) < 3:
            raise ValueError(f"'{column}' 의 유효한 수치가 {len(s)}개뿐입니다.")
        return s, n_bad

    def _check_columns(self, *columns: str) -> None:
        missing = [c for c in columns if c not in self.df.columns]
        if missing:
            raise KeyError(f"{missing} 열이 없습니다. 있는 열: {list(self.df.columns)}")

    # -- 도구 등록 / Registering the tools ---------------------------------
    def _register_all(self) -> None:
        import numpy as np
        import pandas as pd
        from scipy import stats

        # ---------------------------------------------------------------
        @self.tool(
            "list_columns",
            "데이터의 열 이름, 타입, 결측 수, 고유값 수를 돌려준다. 항상 먼저 호출할 것.",
            {},
        )
        def _list_columns() -> dict:
            rows = []
            for c in self.df.columns:
                s = self.df[c]
                num = pd.to_numeric(s, errors="coerce")
                rows.append({
                    "column": c,
                    "dtype": str(s.dtype),
                    "n_missing": int(s.isna().sum()),
                    "n_unique": int(s.nunique(dropna=True)),
                    "numeric_ok": bool(num.notna().sum() > len(s) * 0.8),
                    "sample": [str(v) for v in s.dropna().head(3).tolist()],
                })
            return {"n_rows": int(len(self.df)), "columns": rows}

        # ---------------------------------------------------------------
        @self.tool(
            "describe_column",
            "한 열의 기술통계를 돌려준다. 수치형이면 평균·표준편차·사분위수, 범주형이면 빈도표.",
            {"column": {"type": "str", "desc": "열 이름", "required": True}},
        )
        def _describe_column(column: str) -> dict:
            self._check_columns(column)
            s = self.df[column]
            num = pd.to_numeric(s, errors="coerce")

            if num.notna().sum() > len(s) * 0.8:      # 수치형으로 본다 / treat as numeric
                v = num.dropna()
                q1, q3 = v.quantile([0.25, 0.75])
                return {
                    "column": column, "kind": "numeric", "n": int(len(v)),
                    "n_missing": int(s.isna().sum()),
                    "n_non_numeric": int(num.isna().sum() - s.isna().sum()),
                    "mean": round(float(v.mean()), 4),
                    "sd": round(float(v.std(ddof=1)), 4),
                    "min": round(float(v.min()), 4),
                    "q1": round(float(q1), 4),
                    "median": round(float(v.median()), 4),
                    "q3": round(float(q3), 4),
                    "max": round(float(v.max()), 4),
                    "skew": round(float(v.skew()), 3),
                }
            counts = s.value_counts(dropna=True)
            return {
                "column": column, "kind": "categorical",
                "n": int(s.notna().sum()), "n_missing": int(s.isna().sum()),
                "n_levels": int(counts.size),
                "counts": {str(k): int(v) for k, v in counts.head(15).items()},
            }

        # ---------------------------------------------------------------
        @self.tool(
            "check_normality",
            "한 열의 정규성을 Shapiro-Wilk 로 검정한다. 검정 방법을 고르기 전에 확인할 것.",
            {"column": {"type": "str", "desc": "수치형 열 이름", "required": True},
             "group_column": {"type": "str", "desc": "집단별로 나눠 검정할 열 (선택)"}},
        )
        def _check_normality(column: str, group_column: str | None = None) -> dict:
            if group_column is None:
                v, n_bad = self._numeric(column)
                W, p = stats.shapiro(v[:5000])
                return {"column": column, "n": int(len(v)),
                        "n_non_numeric": n_bad,
                        "statistic_W": round(float(W), 4), "p_value": float(p),
                        "normal_at_0.05": bool(p >= 0.05)}

            self._check_columns(column, group_column)
            out = {}
            work = self.df[[column, group_column]].copy()
            work[column] = pd.to_numeric(work[column], errors="coerce")
            for g, sub in work.dropna().groupby(group_column):
                if len(sub) < 3:
                    out[str(g)] = {"n": int(len(sub)), "error": "표본이 3 미만"}
                    continue
                W, p = stats.shapiro(sub[column][:5000])
                out[str(g)] = {"n": int(len(sub)),
                               "statistic_W": round(float(W), 4),
                               "p_value": float(p),
                               "normal_at_0.05": bool(p >= 0.05)}
            return {"column": column, "by": group_column, "groups": out}

        # ---------------------------------------------------------------
        @self.tool(
            "compare_two_groups",
            "두 독립 집단의 평균(또는 분포)을 비교한다. 등분산은 Welch 로 보정하며, "
            "method=auto 면 정규성 검정 결과에 따라 t검정/Mann-Whitney 를 고른다.",
            {"value_column": {"type": "str", "desc": "비교할 수치형 열", "required": True},
             "group_column": {"type": "str", "desc": "두 수준을 가진 집단 열", "required": True},
             "method": {"type": "str", "enum": ["auto", "t", "mannwhitney"],
                        "desc": "기본 auto"},
             "alpha": {"type": "float", "desc": "유의수준, 기본 0.05"}},
        )
        def _compare_two_groups(value_column: str, group_column: str,
                                method: str = "auto", alpha: float = 0.05) -> dict:
            self._check_columns(value_column, group_column)
            work = self.df[[value_column, group_column]].copy()
            work[value_column] = pd.to_numeric(work[value_column], errors="coerce")
            work = work.dropna()

            levels = sorted(work[group_column].astype(str).unique())
            if len(levels) != 2:
                raise ValueError(
                    f"'{group_column}' 의 수준이 {len(levels)}개입니다 ({levels[:5]}). "
                    f"두 집단 비교에는 2개여야 합니다. 3개 이상이면 compare_many_groups 를 쓰세요."
                )
            a = work[work[group_column].astype(str) == levels[0]][value_column]
            b = work[work[group_column].astype(str) == levels[1]][value_column]

            chosen = method
            if method == "auto":
                pa = stats.shapiro(a[:5000]).pvalue if len(a) >= 3 else 0.0
                pb = stats.shapiro(b[:5000]).pvalue if len(b) >= 3 else 0.0
                chosen = "t" if (pa >= 0.05 and pb >= 0.05) else "mannwhitney"

            if chosen == "t":
                res = stats.ttest_ind(a, b, equal_var=False)      # Welch
                test_name = "Welch independent t-test"
                # Hedges 보정 없는 Cohen's d (pooled sd)
                # EN — Cohen's d with pooled sd, no Hedges correction
                sp = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1))
                             / (len(a) + len(b) - 2))
                effect = {"name": "Cohen's d", "value": round(float((a.mean() - b.mean()) / sp), 4)}
            else:
                res = stats.mannwhitneyu(a, b, alternative="two-sided")
                test_name = "Mann-Whitney U"
                # rank-biserial correlation
                r = 1 - (2 * res.statistic) / (len(a) * len(b))
                effect = {"name": "rank-biserial r", "value": round(float(r), 4)}

            return {
                "test": test_name,
                "method_requested": method, "method_used": chosen,
                "groups": {levels[0]: {"n": int(len(a)), "mean": round(float(a.mean()), 4),
                                       "sd": round(float(a.std(ddof=1)), 4),
                                       "median": round(float(a.median()), 4)},
                           levels[1]: {"n": int(len(b)), "mean": round(float(b.mean()), 4),
                                       "sd": round(float(b.std(ddof=1)), 4),
                                       "median": round(float(b.median()), 4)}},
                "statistic": round(float(res.statistic), 4),
                "p_value": float(res.pvalue),
                "alpha": alpha,
                "significant": bool(res.pvalue < alpha),
                "effect_size": effect,
            }

        # ---------------------------------------------------------------
        @self.tool(
            "compare_paired",
            "같은 대상에서 두 번 측정한 값(전/후)을 비교한다. 대응표본 t검정 또는 Wilcoxon.",
            {"before_column": {"type": "str", "desc": "이전 측정값 열", "required": True},
             "after_column": {"type": "str", "desc": "이후 측정값 열", "required": True},
             "method": {"type": "str", "enum": ["auto", "t", "wilcoxon"]},
             "alpha": {"type": "float"}},
        )
        def _compare_paired(before_column: str, after_column: str,
                            method: str = "auto", alpha: float = 0.05) -> dict:
            self._check_columns(before_column, after_column)
            work = self.df[[before_column, after_column]].apply(
                lambda c: pd.to_numeric(c, errors="coerce")).dropna()
            if len(work) < 3:
                raise ValueError(f"두 열이 모두 있는 행이 {len(work)}개뿐입니다.")
            before, after = work[before_column], work[after_column]
            diff = after - before

            chosen = method
            if method == "auto":
                chosen = "t" if stats.shapiro(diff[:5000]).pvalue >= 0.05 else "wilcoxon"

            if chosen == "t":
                res = stats.ttest_rel(before, after)
                test_name = "paired t-test"
                effect = {"name": "Cohen's dz",
                          "value": round(float(diff.mean() / diff.std(ddof=1)), 4)}
            else:
                res = stats.wilcoxon(before, after)
                test_name = "Wilcoxon signed-rank"
                effect = {"name": "median difference", "value": round(float(diff.median()), 4)}

            return {
                "test": test_name, "method_used": chosen,
                "n_pairs": int(len(work)),
                "mean_before": round(float(before.mean()), 4),
                "mean_after": round(float(after.mean()), 4),
                "mean_difference": round(float(diff.mean()), 4),
                "statistic": round(float(res.statistic), 4),
                "p_value": float(res.pvalue),
                "alpha": alpha, "significant": bool(res.pvalue < alpha),
                "effect_size": effect,
            }

        # ---------------------------------------------------------------
        @self.tool(
            "compare_many_groups",
            "세 집단 이상의 평균(또는 분포)을 비교한다. 일원분산분석 또는 Kruskal-Wallis.",
            {"value_column": {"type": "str", "required": True},
             "group_column": {"type": "str", "required": True},
             "method": {"type": "str", "enum": ["auto", "anova", "kruskal"]},
             "alpha": {"type": "float"}},
        )
        def _compare_many_groups(value_column: str, group_column: str,
                                 method: str = "auto", alpha: float = 0.05) -> dict:
            self._check_columns(value_column, group_column)
            work = self.df[[value_column, group_column]].copy()
            work[value_column] = pd.to_numeric(work[value_column], errors="coerce")
            work = work.dropna()

            groups = [g[value_column].values for _, g in work.groupby(group_column)]
            labels = [str(k) for k, _ in work.groupby(group_column)]
            if len(groups) < 3:
                raise ValueError(
                    f"집단이 {len(groups)}개입니다. 3개 이상이어야 합니다. "
                    f"2개면 compare_two_groups 를 쓰세요."
                )

            chosen = method
            if method == "auto":
                정규 = all(len(g) >= 3 and stats.shapiro(g[:5000]).pvalue >= 0.05 for g in groups)   # normality
                등분산 = stats.levene(*groups).pvalue >= 0.05                                        # equal variance
                chosen = "anova" if (정규 and 등분산) else "kruskal"

            if chosen == "anova":
                res = stats.f_oneway(*groups)
                test_name = "one-way ANOVA"
                grand = np.concatenate(groups)
                ss_b = sum(len(g) * (g.mean() - grand.mean()) ** 2 for g in groups)
                ss_t = ((grand - grand.mean()) ** 2).sum()
                effect = {"name": "eta squared", "value": round(float(ss_b / ss_t), 4)}
            else:
                res = stats.kruskal(*groups)
                test_name = "Kruskal-Wallis"
                n = len(np.concatenate(groups))
                effect = {"name": "epsilon squared",
                          "value": round(float(res.statistic / ((n ** 2 - 1) / (n + 1))), 4)}

            return {
                "test": test_name, "method_used": chosen,
                "groups": {lab: {"n": int(len(g)), "mean": round(float(g.mean()), 4),
                                 "sd": round(float(g.std(ddof=1)), 4)}
                           for lab, g in zip(labels, groups)},
                "statistic": round(float(res.statistic), 4),
                "p_value": float(res.pvalue),
                "alpha": alpha, "significant": bool(res.pvalue < alpha),
                "effect_size": effect,
                "note": "유의하면 사후검정(Tukey 등)이 필요합니다.",
            }

        # ---------------------------------------------------------------
        @self.tool(
            "test_association",
            "두 범주형 변수의 연관성을 카이제곱 검정한다. 기대빈도가 5 미만이면 Fisher 를 권한다.",
            {"column_a": {"type": "str", "required": True},
             "column_b": {"type": "str", "required": True},
             "alpha": {"type": "float"}},
        )
        def _test_association(column_a: str, column_b: str, alpha: float = 0.05) -> dict:
            self._check_columns(column_a, column_b)
            table = pd.crosstab(self.df[column_a], self.df[column_b])
            if table.size < 4:
                raise ValueError("분할표가 2x2 보다 작습니다.")
            chi2, p, dof, expected = stats.chi2_contingency(table)
            min_exp = float(expected.min())
            n = int(table.values.sum())
            cramers_v = float(np.sqrt(chi2 / (n * (min(table.shape) - 1))))
            return {
                "test": "chi-square test of independence",
                "table": table.to_dict(),
                "chi2": round(float(chi2), 4), "df": int(dof),
                "p_value": float(p), "alpha": alpha,
                "significant": bool(p < alpha),
                "min_expected_count": round(min_exp, 2),
                "effect_size": {"name": "Cramer's V", "value": round(cramers_v, 4)},
                "warning": (None if min_exp >= 5 else
                            "기대빈도가 5 미만인 칸이 있습니다. Fisher 정확검정을 고려하세요."),
            }

        # ---------------------------------------------------------------
        @self.tool(
            "correlate",
            "두 수치형 변수의 상관을 구한다. method=auto 면 정규성에 따라 pearson/spearman 선택.",
            {"column_a": {"type": "str", "required": True},
             "column_b": {"type": "str", "required": True},
             "method": {"type": "str", "enum": ["auto", "pearson", "spearman"]}},
        )
        def _correlate(column_a: str, column_b: str, method: str = "auto") -> dict:
            self._check_columns(column_a, column_b)
            work = self.df[[column_a, column_b]].apply(
                lambda c: pd.to_numeric(c, errors="coerce")).dropna()
            if len(work) < 3:
                raise ValueError(f"두 열이 모두 있는 행이 {len(work)}개뿐입니다.")
            a, b = work[column_a], work[column_b]

            chosen = method
            if method == "auto":
                정규 = (stats.shapiro(a[:5000]).pvalue >= 0.05
                        and stats.shapiro(b[:5000]).pvalue >= 0.05)
                chosen = "pearson" if 정규 else "spearman"

            fn = stats.pearsonr if chosen == "pearson" else stats.spearmanr
            res = fn(a, b)
            return {
                "method_used": chosen, "n": int(len(work)),
                "coefficient": round(float(res[0]), 4),
                "p_value": float(res[1]),
                "note": "상관은 인과가 아닙니다.",
            }


In [ ]:
%%writefile react.py
"""전산실험2 6주차 : ReAct 루프 — 에이전트의 심장.

    Thought  → 지금 무엇을 알아야 하는가
    Action   → 어떤 도구를 어떤 인자로 부를 것인가
    Observation → 도구가 돌려준 결과 (실패도 관찰이다)
        ↓ 목표가 달성될 때까지 반복

이 구조는 3주차 `ask_json()` 의 되먹임 루프와 같다.
달라진 것은 "관찰"이 파싱 결과가 아니라 **도구 실행 결과**라는 점뿐이다.

설계에서 중요한 것 세 가지:
    1. 종료 조건을 여러 겹으로 둔다 (정답 / 스텝 상한 / 연속 실패 / 반복 감지)
    2. 실패를 루프 밖으로 던지지 않는다
    3. 모든 단계를 기록한다 (Trace) — 11주차 평가의 재료가 된다

EN — Week 6: the ReAct loop, the heart of an agent.
    Thought      → what do I need to know right now
    Action       → which tool do I call, with which arguments
    Observation  → what the tool returned (a failure is an observation too)
        ↓ repeat until the goal is met
This is the same shape as the feedback loop in week 3's `ask_json()`; the only
change is that the "observation" is now a tool result rather than a parse
result. Three things matter in the design: (1) layer several stop conditions
(answer found / step limit / consecutive failures / repetition detected),
(2) never let a failure escape the loop, and (3) record every step in a Trace —
that becomes the raw material for the week-11 evaluation.

사용 예 · Example:
    import pandas as pd
    from kmu_llm import LLM
    from tools import StatToolbox
    from react import ReActAgent

    box = StatToolbox(pd.read_csv("sample_clinic.csv"))
    agent = ReActAgent(LLM(), box, max_steps=8)
    trace = agent.run("치료 전후로 혈압이 유의하게 떨어졌나요?")
    print(trace.final_answer)
    trace.show()
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any

__all__ = ["Step", "Trace", "ReActAgent", "REACT_SYSTEM"]


# --------------------------------------------------------------------------
# 실행 기록 / Execution record
# --------------------------------------------------------------------------
@dataclass
class Step:
    """한 사이클의 기록. / The record of one cycle."""

    n: int
    thought: str = ""
    action: str | None = None
    args: dict[str, Any] = field(default_factory=dict)
    observation: str = ""
    ok: bool = False
    raw: str = ""           # 모델의 원문 (디버깅용) / the model's raw reply, for debugging
    error: str | None = None

    def summary(self) -> str:
        head = f"[{self.n}] {self.action}({json.dumps(self.args, ensure_ascii=False)})"
        mark = "OK" if self.ok else "실패"
        return f"{head}  -> {mark}"


@dataclass
class Trace:
    """한 번의 `run()` 전체 기록.

    11주차에서 이 객체를 모아 성능을 추정한다.

    EN — The full record of a single `run()`. In week 11 we collect these
    objects to estimate the agent's performance.
    """

    goal: str
    steps: list[Step] = field(default_factory=list)
    final_answer: str | None = None
    stop_reason: str = ""
    success: bool = False

    def __len__(self) -> int:
        return len(self.steps)

    @property
    def tools_used(self) -> list[str]:
        return [s.action for s in self.steps if s.action]

    @property
    def n_failed(self) -> int:
        return sum(1 for s in self.steps if not s.ok)

    def show(self, max_obs: int = 220) -> None:
        """사람이 읽는 실행 경로. / The execution path, for a human to read."""
        print(f"목표: {self.goal}")
        print("=" * 70)
        for s in self.steps:
            print(f"[{s.n}] 생각: {s.thought[:150]}")
            print(f"     행동: {s.action}({json.dumps(s.args, ensure_ascii=False)})")
            obs = s.observation.replace("\n", " ")
            print(f"     관찰: {'OK  ' if s.ok else '실패 '}{obs[:max_obs]}")
            print()
        print("=" * 70)
        print(f"종료 사유: {self.stop_reason}  |  스텝 {len(self.steps)}  |  실패 {self.n_failed}")
        print(f"최종 답변: {self.final_answer}")

    def to_dict(self) -> dict[str, Any]:
        return {
            "goal": self.goal,
            "steps": [
                {"n": s.n, "thought": s.thought, "action": s.action,
                 "args": s.args, "ok": s.ok, "observation": s.observation[:500]}
                for s in self.steps
            ],
            "final_answer": self.final_answer,
            "stop_reason": self.stop_reason,
            "success": self.success,
        }


# --------------------------------------------------------------------------
# 프롬프트 / Prompt
# --------------------------------------------------------------------------
REACT_SYSTEM = """\
너는 통계 분석 에이전트다. 도구를 하나씩 사용해 사용자의 질문에 답한다.

절대 규칙:
- 너는 계산하지 않는다. 평균, 표준편차, p값은 반드시 도구로 구한다.
- 한 번에 도구 하나만 부른다.
- 도구가 돌려준 값만 사실로 인정한다. 추측한 숫자를 말하지 않는다.

매 단계마다 아래 형식의 JSON 하나만 출력한다. 다른 말은 하지 않는다.

행동할 때:
{"thought": "왜 이 도구가 필요한지 한 문장", "action": "도구이름", "args": {...}}

답이 충분히 모였을 때:
{"thought": "왜 끝내는지 한 문장", "action": "finish", "args": {"answer": "최종 답변"}}

권장 순서:
1. list_columns 로 실제 열 이름과 결측 상황을 먼저 확인한다.
2. 필요하면 describe_column / check_normality 로 데이터를 진단한다.
3. 적절한 검정 도구를 부른다.
4. finish 로 결과를 해석해 답한다. p값과 효과크기를 함께 언급한다."""


def build_prompt(goal: str, toolbox: Any, steps: list[Step], max_steps: int) -> str:
    """이번 단계에 보낼 user 메시지를 만든다.

    EN — Build the user message to send for this step.
    """
    parts = [
        f"## 사용 가능한 도구\n{toolbox.describe()}\n",
        f'## 질문\n{goal}\n',
    ]

    if steps:
        parts.append("## 지금까지의 진행")
        for s in steps:
            parts.append(f"[{s.n}] 생각: {s.thought}")
            parts.append(f"    행동: {s.action}({json.dumps(s.args, ensure_ascii=False)})")
            parts.append(f"    관찰: {s.observation}")
        parts.append("")

    남은 = max_steps - len(steps)
    if 남은 <= 2:
        parts.append(
            f"** 남은 단계가 {남은}번뿐이다. 지금까지 얻은 정보로 finish 하라. **\n"
        )
    parts.append("다음에 할 일을 JSON 하나로 출력하라.")
    return "\n".join(parts)


# --------------------------------------------------------------------------
# 에이전트 / The agent
# --------------------------------------------------------------------------
class ReActAgent:
    """Thought → Action → Observation 을 반복한다.

    EN — Repeats Thought → Action → Observation until it is done.

    Parameters
    ----------
    llm : LLM 인스턴스 / an LLM instance
    toolbox : ToolRegistry (보통 StatToolbox) / usually a StatToolbox
    max_steps : 스텝 상한. 무한 루프 방지
                EN — step ceiling, to prevent an infinite loop
    max_consecutive_failures : 연속 실패가 이만큼이면 포기
                EN — give up after this many consecutive failures
    detect_repeat : 같은 (도구, 인자) 를 반복하면 경고를 관찰로 넣는다
                EN — if the same (tool, args) repeats, feed a warning back as
                the observation
    verbose : 진행 상황 출력 / print progress
    """

    def __init__(
        self,
        llm: Any,
        toolbox: Any,
        *,
        max_steps: int = 8,
        max_consecutive_failures: int = 3,
        detect_repeat: bool = True,
        verbose: bool = True,
        system: str = REACT_SYSTEM,
    ) -> None:
        self.llm = llm
        self.toolbox = toolbox
        self.max_steps = max_steps
        self.max_consecutive_failures = max_consecutive_failures
        self.detect_repeat = detect_repeat
        self.verbose = verbose
        self.system = system

    # ----------------------------------------------------------------
    def run(self, goal: str, **kw: Any) -> Trace:
        """목표를 받아 끝까지 돌린다. **예외를 밖으로 던지지 않는다.**

        EN — Take a goal and run to completion. IT NEVER RAISES.
        """
        from structured import JSONParseError, parse_json

        trace = Trace(goal=goal)
        연속실패 = 0
        본적있는행동: set[str] = set()

        kw.setdefault("temperature", 0.0)
        kw.setdefault("max_tokens", 600)

        while len(trace.steps) < self.max_steps:
            n = len(trace.steps) + 1
            step = Step(n=n)

            # --- 1) 생각하고 행동을 정한다 / Think and choose an action ---
            prompt = build_prompt(goal, self.toolbox, trace.steps, self.max_steps)
            try:
                step.raw = self.llm.chat(
                    [{"role": "system", "content": self.system},
                     {"role": "user", "content": prompt}], **kw
                )
            except Exception as exc:
                step.error = f"모델 호출 실패: {type(exc).__name__}: {exc}"
                step.observation = step.error
                trace.steps.append(step)
                연속실패 += 1
                if 연속실패 >= self.max_consecutive_failures:
                    trace.stop_reason = "모델 호출이 계속 실패함"
                    break
                continue

            try:
                결정 = parse_json(step.raw)
                if not isinstance(결정, dict):
                    raise JSONParseError("최상위가 객체가 아닙니다.")
            except JSONParseError as exc:
                step.error = str(exc)
                step.observation = (
                    f"[형식 오류] 응답을 JSON으로 읽을 수 없다: {exc}\n"
                    '{"thought": ..., "action": ..., "args": {...}} 형식으로만 출력하라.'
                )
                trace.steps.append(step)
                연속실패 += 1
                if self.verbose:
                    print(f"[{n}] 형식 오류 -> 되먹임")
                if 연속실패 >= self.max_consecutive_failures:
                    trace.stop_reason = "형식 오류가 계속됨"
                    break
                continue

            step.thought = str(결정.get("thought", ""))[:400]
            step.action = 결정.get("action")
            step.args = 결정.get("args") or {}
            if not isinstance(step.args, dict):
                step.args = {}

            if self.verbose:
                print(f"[{n}] 생각: {step.thought[:90]}")
                print(f"     행동: {step.action}({json.dumps(step.args, ensure_ascii=False)[:110]})")

            # --- 2) 종료 행동인가 / Is this the stop action? --------------
            if step.action == "finish":
                step.ok = True
                step.observation = "(종료)"
                trace.steps.append(step)
                trace.final_answer = str(step.args.get("answer", "")).strip()
                trace.stop_reason = "모델이 finish 를 선택"
                trace.success = bool(trace.final_answer)
                if self.verbose:
                    print(f"     -> 종료\n")
                break

            # --- 3) 같은 행동을 반복하는가 / Is it repeating itself? ------
            열쇠 = f"{step.action}|{json.dumps(step.args, ensure_ascii=False, sort_keys=True)}"
            if self.detect_repeat and 열쇠 in 본적있는행동:
                step.ok = False
                step.observation = (
                    "[반복 감지] 방금 전과 똑같은 도구를 똑같은 인자로 다시 불렀다. "
                    "이미 그 결과는 위에 있다. 다른 도구를 쓰거나 finish 하라."
                )
                trace.steps.append(step)
                연속실패 += 1
                if self.verbose:
                    print("     -> 반복 감지\n")
                if 연속실패 >= self.max_consecutive_failures:
                    trace.stop_reason = "같은 행동을 반복함"
                    break
                continue
            본적있는행동.add(열쇠)

            # --- 4) 도구를 실행하고 관찰한다 / Run the tool and observe ---
            result = self.toolbox.call(step.action, step.args)
            step.ok = result.ok
            step.observation = result.observation()
            step.error = result.error
            trace.steps.append(step)

            if self.verbose:
                print(f"     관찰: {'OK' if step.ok else '실패'} "
                      f"{step.observation.replace(chr(10), ' ')[:110]}\n")

            연속실패 = 0 if step.ok else 연속실패 + 1
            if 연속실패 >= self.max_consecutive_failures:
                trace.stop_reason = f"도구 실행이 {연속실패}회 연속 실패"
                break

        # --- 루프를 다 돌았는데 finish 가 없었다면 -----------------------
        # EN — the loop ended without the model ever choosing finish
        if trace.final_answer is None and not trace.stop_reason:
            trace.stop_reason = f"스텝 상한({self.max_steps}) 도달"

        if trace.final_answer is None:
            trace.final_answer = self._force_answer(trace, **kw)

        return trace

    # ----------------------------------------------------------------
    def _force_answer(self, trace: Trace, **kw: Any) -> str:
        """finish 없이 끝났을 때, 지금까지의 관찰로 답을 만들어 본다.

        빈손으로 끝내는 것보다는 낫지만, **성공으로 치지는 않는다.**
        `trace.success` 는 False 로 남는다.

        EN — When the run ended without finish, try to compose an answer from
        the observations gathered so far. Better than coming back empty-handed,
        but it DOES NOT COUNT AS SUCCESS: `trace.success` stays False.
        """
        성공관찰 = [s.observation for s in trace.steps if s.ok]
        if not 성공관찰:
            return "(도구 실행에 모두 실패해 답할 수 없습니다.)"

        모음 = "\n\n".join(성공관찰)[:3000]
        try:
            return self.llm.ask(
                f"질문: {trace.goal}\n\n"
                f"아래는 도구 실행 결과다. 이것만 근거로 답하라. "
                f"근거가 부족하면 부족하다고 말하라. 숫자를 지어내지 마라.\n\n{모음}",
                system="너는 통계 분석 결과를 해석하는 도구다.",
                temperature=0.0,
                max_tokens=kw.get("max_tokens", 600),
            )
        except Exception as exc:
            return f"(답변 생성 실패: {type(exc).__name__})"


In [ ]:
%%writefile executor.py
"""전산실험2 7주차 : 코드 실행 에이전트.

도구를 미리 만들어 두는 대신, 모델이 분석 코드를 **직접 써서 실행**한다.
오류가 나면 Traceback 을 되먹여 고치게 한다.

    코드 생성 → 검사 → 실행 → 결과 관찰 → 실패면 Traceback 되먹임
                                              ↑ 3주차·6주차와 같은 구조

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 안전에 관한 정직한 경고

 여기 있는 `SafeExecutor` 는 **진짜 샌드박스가 아니다.**
 AST 검사와 제한된 네임스페이스는 "실수로 사고 치는 것"을 막는 장치이지,
 작정한 공격을 막는 장치가 아니다. 파이썬에서 언어 수준의 완전한 격리는
 사실상 불가능하다는 것이 알려져 있다.

 따라서 이 코드는 다음 전제에서만 쓴다.
   - 실행 환경이 **격리된 일회용 VM** 이다 (Colab 런타임이 그렇다)
   - 코드를 만드는 주체가 **우리가 통제하는 모델**이다
   - 그 안에 **중요한 자격증명이 없다**

 실제 서비스에서 사용자 입력으로 코드를 실행해야 한다면 컨테이너,
 gVisor, Firecracker 같은 OS 수준 격리를 쓴다. 파이썬 코드로 막지 않는다.

 EN — An honest warning about safety. `SafeExecutor` below is NOT A REAL
 SANDBOX. The AST check and the restricted namespace stop accidents, not a
 determined attack; complete language-level isolation in Python is known to be
 practically impossible. So use this code only under these assumptions: the
 runtime is an isolated, disposable VM (a Colab runtime is), the code is
 produced by a model we control, and there are no valuable credentials inside.
 In a real service that runs user-supplied code, use OS-level isolation —
 containers, gVisor, Firecracker — not Python code.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

EN — Week 7: the code-execution agent. Instead of pre-built tools, the model
WRITES AND RUNS the analysis code itself; when it fails, the traceback is fed
back so it can fix it. Generate → check → run → observe → feed the traceback
back on failure: the same structure as weeks 3 and 6.

사용 예 · Example:
    import pandas as pd
    from kmu_llm import LLM
    from executor import SafeExecutor, CodeAgent

    ex = SafeExecutor({"df": pd.read_csv("sample_clinic.csv")})
    print(ex.run("result = df['bmi'].mean()").summary())

    agent = CodeAgent(LLM(), ex)
    out = agent.run("치료군과 대조군의 BMI 평균을 비교하고 p값을 구하라")
    print(out.result)
"""

from __future__ import annotations

import ast
import contextlib
import io
import signal
import traceback
from dataclasses import dataclass, field
from typing import Any

__all__ = [
    "ALLOWED_IMPORTS",
    "CodeCheckError",
    "ExecResult",
    "SafeExecutor",
    "CodeAgent",
    "CodeRun",
]

# 허용할 모듈. 화이트리스트 방식 — 목록에 없으면 막는다.
# 블랙리스트("os 를 막자")는 반드시 빠뜨리는 것이 생긴다.
# EN — Modules we allow. A whitelist: anything not listed is blocked. A
#      blacklist ("let's block os") always misses something.
ALLOWED_IMPORTS = {
    "numpy", "np",
    "pandas", "pd",
    "scipy", "scipy.stats",
    "statsmodels", "statsmodels.api", "statsmodels.formula.api",
    "statsmodels.stats.multicomp", "statsmodels.stats.proportion",
    "sklearn", "sklearn.linear_model", "sklearn.metrics",
    "matplotlib", "matplotlib.pyplot",
    "math", "statistics", "itertools", "collections", "json", "re",
}

# 이름만 나와도 막을 것들 / Names that are blocked on sight
FORBIDDEN_NAMES = {
    "__import__", "eval", "exec", "compile", "open", "input",
    "globals", "locals", "vars", "getattr", "setattr", "delattr",
    "exit", "quit", "breakpoint", "memoryview",
}

# 속성 접근으로 우회하는 전형적인 수법 / The classic attribute-access escapes
FORBIDDEN_ATTRS = {
    "__globals__", "__builtins__", "__subclasses__", "__bases__",
    "__mro__", "__class__", "__code__", "__closure__", "__reduce__",
    "__getattribute__", "__dict__", "__loader__", "__spec__",
}


class CodeCheckError(ValueError):
    """실행 전 검사에서 걸렸을 때.

    EN — Raised when the pre-execution check rejects the code.
    """


# --------------------------------------------------------------------------
# 정적 검사 / Static checking
# --------------------------------------------------------------------------
def check_code(source: str) -> list[str]:
    """실행하기 전에 코드를 훑어 위험한 것을 찾는다.

    파싱해서 AST 를 보는 이유: 문자열 검색(`"os" in code`)은
    `"o" + "s"` 같은 것에 뚫린다. 구문 트리는 그런 속임수에 강하다.

    EN — Scan the code for dangerous constructs before running it. We parse to
    an AST rather than searching the text, because a string search (`"os" in
    code`) is defeated by `"o" + "s"`; a syntax tree is not.

    Returns
    -------
    list[str] : 문제 목록 (빈 목록이면 통과) / problems; empty means it passed
    """
    try:
        tree = ast.parse(source)
    except SyntaxError as exc:
        return [f"문법 오류: {exc.msg} ({exc.lineno}행)"]

    problems: list[str] = []

    for node in ast.walk(tree):
        # import 검사 / import checks
        if isinstance(node, ast.Import):
            for alias in node.names:
                root = alias.name.split(".")[0]
                if alias.name not in ALLOWED_IMPORTS and root not in ALLOWED_IMPORTS:
                    problems.append(
                        f"'{alias.name}' 모듈은 허용되지 않습니다. "
                        f"허용 목록: {sorted(ALLOWED_IMPORTS)}"
                    )
        elif isinstance(node, ast.ImportFrom):
            mod = node.module or ""
            root = mod.split(".")[0]
            if mod not in ALLOWED_IMPORTS and root not in ALLOWED_IMPORTS:
                problems.append(f"'{mod}' 모듈은 허용되지 않습니다.")

        # 금지된 이름 호출 / calls to forbidden names
        elif isinstance(node, ast.Name) and node.id in FORBIDDEN_NAMES:
            problems.append(f"'{node.id}' 는 사용할 수 없습니다.")

        # 던더 속성 접근 (샌드박스 탈출의 전형적 경로)
        # EN — dunder attribute access, the classic sandbox-escape route
        elif isinstance(node, ast.Attribute) and node.attr in FORBIDDEN_ATTRS:
            problems.append(f"'.{node.attr}' 속성 접근은 허용되지 않습니다.")

    return problems


# --------------------------------------------------------------------------
# 실행 결과 / Execution result
# --------------------------------------------------------------------------
@dataclass
class ExecResult:
    """코드 한 번 실행의 결과. / The result of running the code once."""

    ok: bool
    code: str
    stdout: str = ""
    result: Any = None            # 코드가 `result` 변수에 담은 값 / what the code put in `result`
    error: str | None = None      # 예외 타입과 메시지 / exception type and message
    traceback: str = ""           # 모델에게 되먹일 전체 Traceback / traceback fed back to the model
    blocked: list[str] = field(default_factory=list)   # 검사에서 걸린 이유 / why the check rejected it
    figures: int = 0              # 그려진 그림 수 / number of figures drawn

    def observation(self, max_chars: int = 1500) -> str:
        """모델에게 돌려줄 관찰 문자열.

        EN — The observation string handed back to the model.
        """
        if self.blocked:
            return ("[실행 거부] 코드가 안전 검사를 통과하지 못했습니다:\n"
                    + "\n".join(f"- {b}" for b in self.blocked)
                    + "\n허용된 모듈만 쓰고, 파일·시스템 접근을 하지 마세요.")

        if not self.ok:
            tb = self.traceback[-max_chars:] if self.traceback else str(self.error)
            return f"[실행 실패]\n{tb}\n\n위 오류를 고쳐서 코드를 다시 작성하세요."

        parts = []
        if self.stdout.strip():
            parts.append(f"[출력]\n{self.stdout.strip()[:max_chars]}")
        if self.result is not None:
            parts.append(f"[result]\n{str(self.result)[:max_chars]}")
        if self.figures:
            parts.append(f"[그림 {self.figures}개 생성됨]")
        if not parts:
            parts.append("[실행 성공했지만 출력이 없습니다. "
                         "print() 를 쓰거나 result 변수에 값을 담으세요.]")
        return "\n\n".join(parts)

    def summary(self) -> str:
        if self.blocked:
            return f"거부됨: {self.blocked[0]}"
        if not self.ok:
            return f"실패: {self.error}"
        return f"성공 (출력 {len(self.stdout)}자, result={type(self.result).__name__})"


# --------------------------------------------------------------------------
# 실행기 / The executor
# --------------------------------------------------------------------------
class SafeExecutor:
    """제한된 네임스페이스에서 코드를 실행한다.

    **진짜 샌드박스가 아니다.** 모듈 상단의 경고를 읽을 것.

    EN — Runs code in a restricted namespace. THIS IS NOT A REAL SANDBOX; read
    the warning at the top of the module.

    Parameters
    ----------
    context : 코드에서 쓸 수 있게 미리 넣어 줄 변수들 (보통 {"df": DataFrame})
              EN — variables pre-loaded for the code, usually {"df": DataFrame}
    timeout : 초 단위 상한. POSIX(Colab 포함)에서만 동작한다.
              EN — limit in seconds; works on POSIX only (Colab included)
    persist : True 면 실행 사이에 변수가 유지된다 (여러 단계 분석에 유용)
              EN — True keeps variables between runs, useful for multi-step work
    """

    def __init__(
        self,
        context: dict[str, Any] | None = None,
        *,
        timeout: int = 30,
        persist: bool = True,
    ) -> None:
        self.timeout = timeout
        self.persist = persist
        self.context = dict(context or {})
        self.namespace: dict[str, Any] = {}
        self.history: list[ExecResult] = []
        self.reset()

    # ----------------------------------------------------------------
    def reset(self) -> None:
        """네임스페이스를 초기 상태로 되돌린다.

        EN — Return the namespace to its initial state.
        """
        import numpy as np
        import pandas as pd
        from scipy import stats

        ns: dict[str, Any] = {
            "np": np, "numpy": np,
            "pd": pd, "pandas": pd,
            "stats": stats,
        }
        try:
            import matplotlib

            matplotlib.use("Agg")           # 화면 없이 그림만 만든다 / render figures headlessly
            import matplotlib.pyplot as plt

            ns["plt"] = plt
        except Exception:
            pass

        ns.update(self.context)
        ns["result"] = None
        self.namespace = ns

    def add_context(self, **kw: Any) -> None:
        self.context.update(kw)
        self.namespace.update(kw)

    # ----------------------------------------------------------------
    def run(self, code: str) -> ExecResult:
        """코드를 검사하고 실행한다. **예외를 밖으로 던지지 않는다.**

        EN — Check the code and run it. IT NEVER RAISES.
        """
        problems = check_code(code)
        if problems:
            res = ExecResult(ok=False, code=code, blocked=problems,
                             error="안전 검사 실패")
            self.history.append(res)
            return res

        if not self.persist:
            self.reset()
        self.namespace["result"] = None

        n_fig_before = self._count_figures()
        buf = io.StringIO()
        res = ExecResult(ok=False, code=code)

        try:
            with self._time_limit(self.timeout):
                with contextlib.redirect_stdout(buf):
                    exec(compile(code, "<agent_code>", "exec"), self.namespace)
            res.ok = True
            res.result = self.namespace.get("result")
        except TimeoutError as exc:
            res.error = str(exc)
            res.traceback = str(exc)
        except Exception as exc:
            res.error = f"{type(exc).__name__}: {exc}"
            res.traceback = self._clean_traceback(code)
        finally:
            res.stdout = buf.getvalue()
            res.figures = max(0, self._count_figures() - n_fig_before)

        self.history.append(res)
        return res

    # ----------------------------------------------------------------
    @staticmethod
    def _count_figures() -> int:
        try:
            import matplotlib.pyplot as plt

            return len(plt.get_fignums())
        except Exception:
            return 0

    @staticmethod
    def _clean_traceback(code: str = "") -> str:
        """모델이 고쳐야 할 부분만 남긴 Traceback 을 만든다.

        그냥 `format_exc()` 를 주면 실행기 내부 프레임과 pandas·numpy 내부
        프레임이 잔뜩 딸려 온다. 모델은 그것을 고칠 수 없고, 토큰만 먹고,
        정작 자기 코드의 몇 번째 줄이 문제인지는 묻힌다.

        그래서 **`<agent_code>` 프레임만** 남기고 마지막 예외 메시지를 붙인다.

        EN — Build a traceback containing only what the model can actually fix.
        A plain `format_exc()` drags in executor frames and pandas/numpy
        internals: the model cannot fix those, they burn tokens, and they bury
        the line of its own code that failed. So we keep only `<agent_code>`
        frames and append the final exception message.
        """
        import sys as _sys

        exc_type, exc_value, tb = _sys.exc_info()
        if exc_type is None:
            return ""

        코드줄 = code.splitlines()
        프레임 = []
        for frame, lineno in traceback.walk_tb(tb):
            if frame.f_code.co_filename != "<agent_code>":
                continue                      # 라이브러리·실행기 내부는 버린다 / drop library and executor frames
            본문 = 코드줄[lineno - 1].strip() if 0 < lineno <= len(코드줄) else ""
            프레임.append(f"  {lineno}행:  {본문}")

        머리 = "Traceback (내가 작성한 코드 안에서):"
        꼬리 = "".join(traceback.format_exception_only(exc_type, exc_value)).strip()

        if not 프레임:
            # 문법 오류 등으로 프레임이 안 잡히는 경우
            # EN — no frames captured, e.g. for a syntax error
            return 꼬리
        return "\n".join([머리, *프레임, "", 꼬리])

    @staticmethod
    @contextlib.contextmanager
    def _time_limit(seconds: int):
        """POSIX 에서만 동작하는 실행 시간 제한.

        Windows 에는 `SIGALRM` 이 없어 그냥 통과시킨다.
        **무한 루프를 만나면 셀이 멈춘다는 뜻이다.** 정직하게 알려 둔다.
        Colab 은 리눅스이므로 수업 중에는 동작한다.

        EN — A wall-clock limit that only works on POSIX. Windows has no
        `SIGALRM`, so the limit is skipped there — which honestly means an
        infinite loop will hang the cell. Colab is Linux, so it works in class.
        """
        if not hasattr(signal, "SIGALRM") or seconds <= 0:
            yield
            return

        def _handler(signum, frame):
            raise TimeoutError(f"실행이 {seconds}초를 넘겨 중단했습니다. "
                               f"무한 루프이거나 계산이 너무 무겁습니다.")

        old = signal.signal(signal.SIGALRM, _handler)
        signal.alarm(seconds)
        try:
            yield
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, old)


# --------------------------------------------------------------------------
# 코드 작성 에이전트 / The code-writing agent
# --------------------------------------------------------------------------
CODE_SYSTEM = """\
너는 통계 분석 코드를 작성하는 도구다. 파이썬 코드만 출력한다.

사용 가능한 것:
- df : 분석 대상 DataFrame (이미 로드되어 있다)
- pd, np, stats (scipy.stats), plt (matplotlib.pyplot)

규칙:
- 코드만 출력한다. 설명, 인사, 마크다운 코드블록 표시를 쓰지 않는다.
- 최종 결과를 반드시 `result` 변수에 담는다. 딕셔너리를 권장한다.
- 중간 과정은 print() 로 보여준다.
- 파일을 읽거나 쓰지 않는다. 인터넷에 접속하지 않는다.
- 허용되지 않은 모듈을 import 하지 않는다.
- 숫자를 지어내지 않는다. 반드시 계산해서 얻는다.

예시:
g1 = df[df['group'] == 'treatment']['bmi'].dropna()
g2 = df[df['group'] == 'control']['bmi'].dropna()
t, p = stats.ttest_ind(g1, g2, equal_var=False)
result = {'test': 'Welch t-test', 'n1': len(g1), 'n2': len(g2),
          'statistic': round(float(t), 4), 'p_value': float(p)}
print(result)"""


@dataclass
class CodeRun:
    """코드 작성 에이전트의 실행 기록.

    EN — The run record of the code-writing agent.
    """

    goal: str
    attempts: list[ExecResult] = field(default_factory=list)
    result: Any = None
    success: bool = False
    stop_reason: str = ""

    @property
    def n_attempts(self) -> int:
        return len(self.attempts)

    @property
    def final_code(self) -> str:
        return self.attempts[-1].code if self.attempts else ""

    def show(self) -> None:
        print(f"목표: {self.goal}")
        print("=" * 70)
        for i, a in enumerate(self.attempts, 1):
            print(f"--- 시도 {i} ---")
            print(a.code[:600])
            print(f">>> {a.summary()}")
            if not a.ok:
                print((a.traceback or "")[:400])
            print()
        print("=" * 70)
        print(f"{self.stop_reason}  |  시도 {self.n_attempts}회  |  성공 {self.success}")
        print(f"결과: {self.result}")


class CodeAgent:
    """코드를 쓰고, 실행하고, 실패하면 Traceback 을 보고 고친다.

    6주차 ReAct 루프와 같은 구조다. 관찰이 '도구 실행 결과' 에서
    '코드 실행 결과' 로 바뀌었을 뿐이다.

    EN — Writes code, runs it, and fixes it from the traceback when it fails.
    Same structure as the week-6 ReAct loop; the observation is now a code
    result instead of a tool result.
    """

    def __init__(
        self,
        llm: Any,
        executor: SafeExecutor,
        *,
        max_attempts: int = 4,
        verbose: bool = True,
        system: str = CODE_SYSTEM,
    ) -> None:
        self.llm = llm
        self.executor = executor
        self.max_attempts = max_attempts
        self.verbose = verbose
        self.system = system

    # ----------------------------------------------------------------
    @staticmethod
    def extract_code(text: str) -> str:
        """모델 응답에서 코드만 뽑아낸다.

        3주차 `extract_json()` 과 같은 발상이다.
        지시해도 코드블록으로 감싸는 일이 흔하다.

        EN — Pull just the code out of the model's reply — same idea as
        `extract_json()` in week 3. Even when told not to, models routinely
        wrap the code in a fenced block.
        """
        import re

        fence = re.search(r"```(?:python|py)?\s*(.*?)```", text, re.DOTALL)
        if fence:
            return fence.group(1).strip()
        return text.strip()

    def _data_hint(self) -> str:
        """df 의 모양을 프롬프트에 넣어 준다.

        5주차에서 확인했듯, 실제 열 이름을 보여주면 오류가 크게 준다.

        EN — Put the shape of `df` into the prompt. As week 5 showed, showing
        the real column names cuts errors sharply.
        """
        df = self.executor.namespace.get("df")
        if df is None:
            return ""
        try:
            정보 = [f"- {c} ({df[c].dtype}, 결측 {int(df[c].isna().sum())})"
                    for c in df.columns]
            head = df.head(3).to_string()
            return (f"\n## 데이터 (df)\n행 {len(df)}개, 열 {len(df.columns)}개\n"
                    + "\n".join(정보) + f"\n\n앞 3행:\n{head}\n")
        except Exception:
            return ""

    # ----------------------------------------------------------------
    def run(self, goal: str, **kw: Any) -> CodeRun:
        """목표를 받아 코드를 쓰고 실행한다. 실패하면 고쳐서 다시.

        EN — Take a goal, write the code and run it; on failure, fix and retry.
        """
        kw.setdefault("temperature", 0.0)
        kw.setdefault("max_tokens", 900)

        run = CodeRun(goal=goal)
        messages = [
            {"role": "system", "content": self.system},
            {"role": "user", "content": f"{self._data_hint()}\n## 요청\n{goal}\n\n코드만 출력하라."},
        ]

        for attempt in range(1, self.max_attempts + 1):
            try:
                원문 = self.llm.chat(messages, **kw)
            except Exception as exc:
                run.stop_reason = f"모델 호출 실패: {type(exc).__name__}"
                return run

            code = self.extract_code(원문)
            if self.verbose:
                print(f"--- 시도 {attempt} ---")
                print(code[:400])

            res = self.executor.run(code)
            run.attempts.append(res)

            if self.verbose:
                print(f">>> {res.summary()}\n")

            if res.ok:
                run.result = res.result
                run.success = True
                run.stop_reason = f"{attempt}회 만에 성공"
                return run

            # 실패 -> Traceback 을 되먹인다 / on failure, feed the traceback back
            messages.append({"role": "assistant", "content": 원문})
            messages.append({"role": "user", "content": res.observation()})

        run.stop_reason = f"{self.max_attempts}회 모두 실패"
        return run


In [ ]:
%%writefile exam_greenhouse.csv
pot_id,fertilizer,greenhouse,yield_before,yield_after,water_liters,temp_c,leaf_count,pest_damage
G0001,C,south,63.63,63.91,2.33,25.8,30.0,No
G0002,C,north,44.55,38.06,2.23,22.1,22.0,no
G0003,A,north,28.47,28.83,2.85,20.5,30.0,no
G0004,B,south,60.01,58.01,0.78,-999,34.0,No
G0005,B,south,68.94,85.16,1.57,25.4,34.0,No
G0006,A,south,41.56,48.18,"2,58",25.1,32.0,no
G0007,A,south,48.84,48.73,2.5,25.0,47.0,No
G0008,A,south,56.79,56.91,1.73,21.4,31.0,No
G0009,A,south,39.05,42.84,2.04,22.7,33.0,No
G0010,A,south,58.79,60.51,2.12,-999,35.0,Yes
G0011,B,north,61.89,71.84,2.01,20.3,27.0,No
G0012,A,north,53.67,61.11,2.89,23.7,32.0,Yes
G0013,A,south,52.87,57.02,3.55,24.6,29.0,Y
G0014,C,north,49.44,43.73,1.72,22.0,49.0,no
G0015,C,south,44.32,51.82,2.98,25.0,36.0,no
G0016,C,north,43.19,41.9,2.32,23.0,26.0,no
G0017,A,south,56.04,56.93,1.97,27.0,28.0,no
G0018,A,south,58.25,68.38,2.25,22.2,22.0,no
G0019,B,south,66.43,81.83,2.32,27.6,33.0,No
G0020,B,north,512.0,67.2,2.56,29.7,27.0,No
G0021,A,south,44.29,42.75,3.57,27.7,32.0,No
G0022,A,north,34.38,34.04,2.06,20.3,24.0,no
G0023,A,south,55.24,62.74,2.59,20.5,33.0,No
G0024,C,north,53.45,60.79,3.19,22.4,26.0,No
G0025,A,south,55.04,61.16,2.89,21.4,26.0,no
G0026,A,south,66.51,66.72,2.26,28.2,36.0,no
G0027,C,north,43.58,50.97,2.79,26.9,26.0,No
G0028,B,south,59.41,67.56,2.62,16.8,30.0,no
G0029,B,north,63.05,69.99,2.97,22.7,36.0,No
G0030,B,north,53.61,60.34,3.61,20.5,37.0,no
G0031,A,north,49.31,50.56,1.38,27.2,34.0,No
G0032,B,south,51.58,61.88,2.52,-999,27.0,No
G0033,B,south,52.52,59.53,2.43,23.6,,No
G0034,B,south,64.49,68.34,2.16,24.9,30.0,no
G0035,C,north,44.66,51.11,2.95,19.3,35.0,no
G0036,C,south,54.42,63.11,3.35,26.2,31.0,No
G0037,C,south,46.66,53.42,1.82,-999,30.0,No
G0038,A,north,47.05,47.01,2.56,22.2,24.0,no
G0039,B,south,69.03,73.57kg,2.6,22.5,19.0,no
G0040,A,south,52.02,56.99,2.2,24.5,34.0,no
G0041,B,north,73.02,83.96,2.35,-999,,No
G0042,B,north,57.19,67.53,2.21,22.3,38.0,No
G0043,C,south,48.89,51.07,3.31,28.4,22.0,no
G0044,C,south,45.51,43.91,2.03,25.5,35.0,No
G0045,C,south,73.1,72.83,2.67,22.4,36.0,no
G0046,A,south,49.77,53.55,2.16,21.1,26.0,no
G0047,A,south,50.02,54.03,1.98,20.2,25.0,no
G0048,B,north,52.5,60.99,2.18,25.8,22.0,No
G0049,C,south,47.19,48.26,2.22,27.9,32.0,No
G0050,C,south,52.92,57.25,"2,28",25.5,39.0,yes
G0051,A,south,52.14,50.21,3.8,24.8,27.0,No
G0052,C,north,58.18,53.26,3.48,21.7,24.0,No
G0053,B,north,60.12,72.63,1.81,21.3,38.0,No
G0054,C,south,48.38,45.24,2.59,-999,17.0,No
G0055,A,south,68.79,71.0,2.87,25.7,27.0,no
G0056,C,south,55.3,54.23,1.96,22.3,40.0,No
G0057,B,north,54.27,59.11,2.5,20.2,30.0,no
G0058,A,north,51.87,51.74,2.79,23.9,39.0,no
G0059,B,south,57.39,72.06,2.1,24.4,26.0,no
G0060,A,north,55.27,59.75,2.22,27.7,31.0,No
G0061,C,south,40.18,46.98,1.55,22.6,34.0,Y
G0062,A,north,42.66,49.74,1.95,25.9,34.0,yes
G0063,C,south,43.26,41.39,2.6,24.1,27.0,No
G0064,A,south,48.12,51.89,1.6,24.6,34.0,No
G0065,B,north,52.48,52.5,1.57,23.5,31.0,No
G0066,A,north,42.64,42.22,2.19,21.4,28.0,No
G0067,B,south,59.28,63.65,"1,91",22.8,37.0,no
G0068,A,north,49.49,51.99,2.58,23.4,40.0,No
G0069,B,south,60.73,65.78,2.81,19.2,31.0,No
G0070,C,north,58.42,63.93,2.28,27.5,30.0,No
G0071,A,south,56.53,55.65,2.68,19.5,40.0,No
G0072,A,north,42.11,44.37,2.54,24.3,31.0,No
G0073,A,north,52.74,51.42,1.51,24.4,23.0,yes
G0074,C,north,50.85,54.03,2.31,25.1,42.0,No
G0075,C,south,47.49,54.63,1.91,27.1,31.0,no
G0076,A,north,43.56,47.87,3.95,22.5,35.0,Yes
G0077,C,south,52.88,49.8,3.15,27.3,37.0,Yes
G0078,B,south,67.53,72.61,2.61,23.4,31.0,yes
G0079,B,north,56.41,60.02,2.77,22.3,31.0,no
G0080,B,north,62.92,58.97,3.46,26.3,28.0,No
G0081,B,south,63.73,66.0,2.28,27.9,36.0,No
G0082,C,south,45.18,46.22,1.4,25.0,28.0,No
G0083,B,south,59.25,67.01,2.2,24.6,30.0,No
G0084,C,south,59.92,67.33,1.85,18.8,30.0,Y
G0085,A,north,52.66,53.71,1.62,25.0,,no
G0086,C,south,47.7,42.38,2.9,28.3,26.0,no
G0087,A,south,60.8,66.25,2.82,27.7,29.0,No
G0088,B,north,68.3,74.46,2.34,27.4,24.0,No
G0089,A,north,41.89,47.62,2.21,24.2,30.0,Yes
G0090,C,north,52.73,55.13,2.61,28.4,29.0,no
G0091,A,north,52.56,56.19,2.46,25.1,35.0,Yes
G0092,B,north,57.11,58.33,1.96,21.5,40.0,yes
G0093,A,north,44.24,47.81,2.67,18.9,28.0,no
G0094,A,south,55.45,57.43,2.66,26.2,27.0,no
G0095,B,north,58.35,64.22,2.01,18.6,44.0,yes
G0096,B,south,59.22,72.91,1.69,23.3,32.0,No
G0097,B,south,56.92,59.72,2.48,25.0,31.0,No
G0098,B,south,71.57,79.2,"3,06",25.6,28.0,No
G0099,C,south,65.59,63.09,2.72,26.9,36.0,yes
G0100,C,south,55.92,53.6,1.83,25.1,37.0,No
G0101,B,north,56.99,55.43,0.61,23.4,28.0,No
G0102,C,south,55.58,53.03,1.95,26.1,21.0,no
G0103,B,south,57.01,69.18,"2,53",23.2,35.0,yes
G0104,B,north,53.63,60.25,1.88,28.0,27.0,No
G0105,C,south,49.91,51.07,2.38,20.6,26.0,No
G0106,C,south,42.83,43.04,1.89,26.9,35.0,yes
G0107,A,north,52.26,47.42,2.62,23.5,29.0,no
G0108,C,north,47.88,49.17,1.72,23.9,36.0,No
G0109,A,south,53.71,55.9,2.51,25.0,31.0,Y
G0110,C,north,46.1,48.32,1.77,25.6,26.0,No
G0111,B,south,57.65,64.51,2.09,23.8,35.0,No
G0112,C,north,54.48,53.27,2.74,25.1,28.0,No
G0113,A,north,54.8,53.37,2.84,23.8,25.0,no
G0114,B,south,61.46,64.15,2.2,24.7,23.0,No
G0115,C,north,37.61,42.91,2.76,24.9,28.0,No
G0116,A,north,47.21,52.37,2.33,30.1,34.0,no
G0117,B,north,56.41,61.96,"3,56",26.9,33.0,Yes
G0118,B,south,64.33,60.61,2.11,22.5,22.0,No
G0119,B,north,55.62,65.12,1.32,26.7,33.0,No
G0120,A,south,56.54,56.58,1.44,21.1,,no
G0121,A,south,55.05,54.88,3.2,21.2,22.0,no
G0122,A,north,49.31,56.34,2.88,26.8,27.0,no
G0123,C,south,61.84,66.2,2.34,25.2,41.0,No
G0124,B,north,48.53,56.65kg,2.04,25.2,25.0,No
G0125,A,south,37.91,39.22,2.66,29.2,30.0,no
G0126,B,north,59.34,65.72,2.84,23.7,33.0,Yes
G0127,B,north,68.35,74.41,2.0,23.3,36.0,No
G0128,A,north,57.22,48.59,1.38,23.7,24.0,no
G0129,A,south,51.69,49.64,3.01,21.3,32.0,No
G0130,C,south,60.8,62.09,1.73,31.1,29.0,No
G0131,B,south,57.66,60.29,2.77,26.8,32.0,no
G0132,A,north,46.24,52.36,2.73,22.2,33.0,yes
G0133,B,north,57.4,55.62,2.07,27.8,27.0,no
G0134,C,south,36.04,45.16,2.53,22.4,32.0,No
G0135,B,south,51.75,58.96,2.59,21.2,30.0,No
G0136,C,south,59.84,58.73,3.11,23.7,30.0,no
G0137,A,south,66.09,60.19,0.98,22.7,33.0,No
G0138,B,south,59.66,63.69,3.35,21.6,41.0,No
G0139,B,south,60.55,64.43,2.05,28.1,22.0,yes
G0140,B,north,68.88,62.12,2.17,17.6,30.0,Yes
G0141,B,south,72.84,78.61,2.87,26.1,36.0,no
G0142,C,north,49.3,52.61,2.61,23.7,40.0,No
G0143,C,north,50.82,55.4,2.04,24.9,28.0,no
G0144,A,south,59.18,61.6,2.45,26.9,32.0,No
G0145,A,south,53.25,64.61,1.31,28.1,900.0,yes
G0146,A,south,52.95,53.67,3.14,21.1,32.0,No
G0147,A,south,50.76,50.9,2.59,27.2,26.0,No
G0148,A,south,64.77,65.06,2.83,23.0,34.0,no
G0149,A,north,45.39,50.96kg,2.27,20.3,34.0,yes
G0150,C,south,46.53,48.82,2.4,23.2,,yes
G0151,A,south,54.04,58.03,1.67,26.2,33.0,No
G0152,B,north,52.18,60.16kg,2.4,21.5,39.0,no
G0153,A,north,40.83,38.31,2.83,19.5,41.0,no
G0154,B,south,64.8,72.95,1.64,24.7,40.0,Yes
G0155,B,north,35.3,54.17,2.3,25.9,34.0,No
G0156,C,north,49.58,46.72,2.66,21.5,25.0,No
G0157,C,south,48.37,51.85,2.61,18.9,25.0,Yes
G0158,A,north,45.83,43.47,2.56,23.4,34.0,Yes
G0159,C,south,41.96,47.06,2.46,27.3,25.0,Yes
G0160,B,north,59.19,63.6,2.72,24.4,34.0,No
G0161,B,south,55.91,67.17,1.38,23.0,30.0,No
G0162,A,north,56.61,60.92,2.78,23.6,29.0,no
G0163,B,north,50.27,57.73,"2,48",24.1,29.0,no
G0164,C,south,59.15,61.58,2.17,24.4,42.0,yes
G0165,B,north,50.53,51.35,2.85,26.0,32.0,No
G0166,B,north,56.1,65.7,2.72,22.1,29.0,no
G0167,A,north,51.51,45.5,1.63,21.5,29.0,no
G0168,A,north,39.29,39.65kg,2.26,-999,26.0,yes
G0169,A,south,45.81,49.61,1.42,17.4,,Yes
G0170,A,south,51.33,60.67,2.61,23.6,25.0,No
G0171,B,north,58.03,65.13,1.56,26.8,33.0,no
G0172,A,south,47.5,41.3,3.7,27.3,28.0,Yes
G0173,C,south,62.38,55.42,1.75,27.3,24.0,no
G0174,A,north,49.02,51.32,2.11,21.8,34.0,No
G0175,A,south,52.3,61.08,1.97,25.7,37.0,No
G0176,B,south,57.87,67.41,"2,91",25.1,22.0,Y
G0177,C,south,68.87,70.77,2.03,28.3,,No
G0178,A,south,57.68,64.22,2.49,29.2,27.0,Yes
G0179,A,north,39.51,44.83,2.64,23.0,27.0,No
G0180,A,north,38.79,37.13,2.65,28.7,39.0,No
G0181,A,north,70.14,65.83kg,2.45,20.8,33.0,No
G0182,C,north,57.1,59.82,1.83,22.4,26.0,No
G0183,B,north,60.78,66.46,2.55,25.1,36.0,Yes
G0184,B,south,67.57,74.94,2.88,-999,37.0,No
G0185,B,north,51.63,51.22,2.99,26.1,37.0,Yes
G0186,C,north,51.58,51.87,2.31,22.7,39.0,No
G0187,A,north,43.53,43.3,2.58,28.3,19.0,No
G0188,B,north,59.97,58.43,3.95,22.9,30.0,No
G0189,A,north,54.2,55.23,2.46,26.9,29.0,No
G0190,C,south,44.75,47.68,2.29,22.7,31.0,No
G0191,B,south,50.15,59.47,2.15,21.5,34.0,no
G0192,A,south,51.78,64.11,2.41,24.5,36.0,no
G0193,A,south,52.26,50.29,2.39,27.6,24.0,No
G0194,B,south,50.26,58.03,2.07,25.5,32.0,No
G0195,B,north,48.43,50.39,2.28,-999,34.0,No
G0196,C,south,47.54,53.9,2.97,26.1,41.0,No
G0197,B,south,58.36,71.86,2.7,-999,32.0,No
G0198,B,north,65.57,71.83,3.17,23.4,18.0,no
G0199,A,south,53.28,54.1,2.63,16.6,23.0,yes
G0200,A,south,45.83,51.69,1.74,24.5,30.0,No
G0201,A,south,50.09,50.18,1.9,23.1,41.0,No
G0202,B,north,63.8,70.25,2.58,-999,31.0,no
G0203,C,south,63.12,57.35,2.77,21.9,33.0,yes
G0204,B,south,57.3,61.12,3.19,24.8,30.0,No
G0205,C,north,38.13,42.29,0.9,18.6,34.0,no
G0206,C,south,51.98,53.82,3.19,25.4,27.0,No
G0207,A,south,52.94,57.02,2.41,25.7,29.0,No
G0208,B,north,56.92,65.96,1.89,25.9,41.0,No
G0209,C,south,65.93,68.41,3.13,20.5,24.0,No
G0210,B,south,57.97,64.46,2.71,26.8,26.0,No


In [ ]:
!pip install -q openai

# 공용 모듈 (1~7주차) / The shared modules from weeks 1-7

# 시험 데이터 / The exam data

from kmu_llm import LLM

PROVIDER = "kmu"        # 서버가 안 되면 "gemini" 로 바꾸세요 (감점 없음)
                        # EN — switch to "gemini" if the server is down; no marks lost
llm = LLM(provider=PROVIDER)
print("준비 완료:", llm)

In [ ]:
import pandas as pd

df = pd.read_csv("exam_greenhouse.csv")
print(df.shape)
df.head()

---
## 1. 데이터를 먼저 보십시오 · Look at the data first

**정리가 필요한 열이 여러 개 있습니다.**
어떤 문제가 있는지 스스로 찾아내는 것이 과제의 일부입니다.

아래 셀들은 탐색을 돕기 위한 것입니다. 자유롭게 고쳐 쓰세요.

> **EN** — **Several columns need cleaning**, and finding out which problems they
> have is part of the assignment. The cells below are there to help you explore —
> change them freely.

In [ ]:
df.info()

In [ ]:
# 각 열의 고유값을 훑어봅니다 / Skim the unique values of every column
for c in df.columns:
    유니크 = df[c].dropna().unique()
    print(f"{c:16s} ({df[c].dtype}) 결측 {df[c].isna().sum():3d}  "
          f"고유 {len(유니크):4d}  예: {list(유니크[:6])}")

In [ ]:
# 수치로 바꿔 보면 무슨 일이 생기는지 확인해 보세요
# EN — See what happens when you coerce each column to numeric.
for c in ["yield_before", "yield_after", "water_liters", "temp_c", "leaf_count"]:
    변환 = pd.to_numeric(df[c], errors="coerce")
    새결측 = int(변환.isna().sum() - df[c].isna().sum())
    print(f"{c:16s} 변환 후 새 결측 {새결측:3d}건   "
          f"min={변환.min()}  max={변환.max()}  mean={변환.mean():.2f}")

> 위 출력에서 **이상해 보이는 것**을 찾으셨습니까?
> 발견한 것을 아래에 적어 두고, 에이전트가 그것을 처리하게 만드십시오.

> **EN** — Did you spot **anything that looks wrong** in the output above? Write
> down what you found below, and make your agent handle it.

In [ ]:
# 발견한 데이터 문제를 여기에 정리하세요 (채점에 반영됩니다)
# EN — List the data problems you found here. This is marked.
발견한_문제 = [
    # 예: "temp_c 에 ... 가 있다",   /  e.g. "temp_c contains ..."
]

for i, p in enumerate(발견한_문제, 1):
    print(f"{i}. {p}")

---
## 2. 답해야 할 다섯 질문 · The five questions to answer

```
Q1. 이 데이터의 각 열은 어떤 성질을 가지는가? 정리가 필요한 열은 무엇인가?
Q2. 처리 전후로 수확량이 유의하게 늘었는가?
Q3. 비료 종류에 따라 처리 후 수확량에 차이가 있는가?
Q4. 비료 종류와 병해 발생 사이에 연관이 있는가?
Q5. 급수량과 수확량 사이에 상관이 있는가?

Q1. What is the nature of each column, and which need cleaning?
Q2. Did yield increase significantly from before treatment to after?
Q3. Does post-treatment yield differ by fertiliser type?
Q4. Is fertiliser type associated with pest damage?
Q5. Is water volume correlated with yield?
```

### 과제 · The task

**질문마다 따로 코드를 짜는 것이 아닙니다.**
질문을 받아 스스로 처리하는 **에이전트 하나**를 만들어 다섯 번 돌리십시오.

> **EN** — **Do not write separate code per question.** Build **one agent** that
> takes a question and handles it, and run it five times.

### 필수 요건 · Requirements

- 1~7주차 부품 중 **최소 세 가지** 사용
  (`ask_json` / `Conversation` / `ToolRegistry` / `ReActAgent` / `SafeExecutor` / `CodeAgent`)
- **데이터 정리가 에이전트 안에** 들어 있을 것 (손으로 미리 정리해 두면 −15점)
- 각 질문의 **실행 경로(Trace) 출력**
- 실패했을 때 **무엇이 왜 실패했는지 보고**

> **EN**
> - Use **at least three** of the week 1–7 components listed above
> - **The data cleaning must live inside the agent** (−15 marks if you clean it by
>   hand beforehand)
> - **Print the execution trace** for each question
> - On failure, **report what failed and why**

In [ ]:
질문들 = [
    "이 데이터의 각 열은 어떤 성질을 가지는가? 정리가 필요한 열은 무엇인가?",
    "처리 전후로 수확량이 유의하게 늘었는가?",
    "비료 종류에 따라 처리 후 수확량에 차이가 있는가?",
    "비료 종류와 병해 발생 사이에 연관이 있는가?",
    "급수량과 수확량 사이에 상관이 있는가?",
]
for i, q in enumerate(질문들, 1):
    print(f"Q{i}. {q}")

---
## 3. 여기부터 여러분의 작업 공간입니다 · Your workspace starts here

아래는 **뼈대만** 있습니다. 자유롭게 고치고, 셀을 추가하십시오.

> **EN** — What follows is **only a skeleton.** Change it freely and add cells.

In [ ]:
# ============================================================
# 데이터 정리 / Data cleaning
# ============================================================
# 힌트: 에이전트가 이것을 '도구'로 부를 수 있게 만들면 좋습니다.
# EN — Hint: it works well to expose this as a *tool* the agent can call.

def 데이터정리(df):
    d = df.copy()
    # 여기에 작성 / write your cleaning here

    return d


정리본 = 데이터정리(df)
print("정리 전:", df.shape, " 정리 후:", 정리본.shape)

In [ ]:
# ============================================================
# 에이전트 구성 / Building the agent
# ============================================================
# 힌트: StatToolbox 를 상속해서 정리 도구를 추가하거나,
#       CodeAgent 에 정리 규칙을 프롬프트로 알려주는 방법이 있습니다.
# EN — Hint: either subclass StatToolbox and add a cleaning tool, or tell
#      CodeAgent the cleaning rules through the prompt.

In [ ]:
# ============================================================
# 다섯 질문 실행 / Running the five questions
# ============================================================
# 각 질문의 실행 경로(Trace)를 반드시 출력하세요.
# EN — You must print the execution trace for every question.

결과모음 = {}

for i, q in enumerate(질문들, 1):
    print("=" * 72)
    print(f"Q{i}. {q}")
    print("=" * 72)

    # 여기에 에이전트 실행 코드 / run your agent here

    print()

---
## 4. 최종 답변 정리 · Writing up the final answers

각 질문에 대한 답을 **사람이 읽을 수 있게** 정리하십시오.

유의성 판정은 이렇게 표현하는 것을 권합니다.

| 상황 | 권장 표현 | 피할 표현 |
|---|---|---|
| p < 0.05 | "유의한 차이가 관찰되었다 (p = ...)" | "차이가 있다" (단정) |
| p ≥ 0.05 | "차이가 있다고 하기 어렵다 (p = ...)" | **"차이가 없다"** |

**EN** — Write the answers up **so a person can read them.** For significance:

| Situation | Recommended | Avoid |
|---|---|---|
| p < 0.05 | "a significant difference was observed (p = ...)" | "there is a difference" (asserted) |
| p ≥ 0.05 | "it is hard to claim a difference (p = ...)" | **"there is no difference"** |

> 마지막 줄이 중요합니다. **유의하지 않다는 것은 "차이가 없다"는 증거가 아닙니다.**
> 이 표현을 정확히 쓰면 가산점이 있습니다.

> **EN** — That last row matters. **Non-significance is not evidence of no
> difference.** Getting this phrasing right earns bonus marks.

In [ ]:
# 최종 답변 (사람이 읽을 형태로) / The final answers, in human-readable form
최종답변 = {
    "Q1": "",
    "Q2": "",
    "Q3": "",
    "Q4": "",
    "Q5": "",
}

for k, v in 최종답변.items():
    print(f"[{k}] {v}\n")

---
## 5. 자기 점검 · Self-check

제출 전에 스스로 확인하십시오.

> **EN** — Check these yourself before submitting.

In [ ]:
점검 = {
    "다섯 질문 모두 답했다": None,
    "데이터 정리가 에이전트 안에 있다": None,
    "1~7주차 부품을 3개 이상 썼다": None,
    "실행 경로(Trace)를 출력했다": None,
    "temp_c 의 -999 를 처리했다": None,
    "yield_after 의 'kg' 를 처리했다": None,
    "water_liters 의 쉼표를 처리했다": None,
    "pest_damage 의 표기 혼용을 처리했다": None,
    "이상치를 확인했다": None,
    "유의하지 않은 결과를 '차이 없음'으로 단정하지 않았다": None,
    "API 키가 코드에 없다": None,
}

# True / False 로 채우세요 / fill each one in with True or False
for k, v in 점검.items():
    표시 = "?" if v is None else ("O" if v else "X")
    print(f"[{표시}] {k}")

미확인 = sum(1 for v in 점검.values() if v is None)
if 미확인:
    print(f"\n{미확인}개 항목이 확인되지 않았습니다.")

---
## 6. 제출 · Submission

```python
# 자기 저장소에 push / push to your own repository
!git clone https://<TOKEN>@github.com/<계정>/<저장소>.git repo
!mkdir -p repo/exam
# 이 노트북을 repo/exam/ 에 저장 (파일 > GitHub에 사본 저장 도 가능)
# EN — save this notebook into repo/exam/ (File > Save a copy in GitHub also works)
```

또는 Colab 메뉴 **파일 → GitHub에 사본 저장** →
저장소 선택, 경로 `exam/중간고사_학번.ipynb`

**제출할 것**: 노트북의 Colab 링크

> **EN** — Or use the Colab menu **File → Save a copy in GitHub**, choose your
> repository, and use the path `exam/중간고사_학번.ipynb`.
> **What to submit**: the notebook's Colab link.

---

## 채점 기준 요약 · Marking criteria

| 항목 | 배점 | 무엇을 보는가 |
|---|---|---|
| 동작 여부 | 30 | 다섯 질문 각 6점. 올바른 검정 + 맞는 수치 + 정확한 해석 |
| 설계 | 25 | 부품을 조합했는가. 만능 도구 하나로 때우지 않았는가 |
| **실패 처리** | **25** | **데이터 함정을 잡았는가. 실패를 조용히 넘기지 않았는가** |
| 가독성·재현성 | 20 | 읽히는가. 다시 돌려도 같은 결과인가. Trace 가 남는가 |

**EN**

| Item | Marks | What is assessed |
|---|---|---|
| Does it work | 30 | 6 per question: the right test, the right numbers, a correct reading |
| Design | 25 | Did you compose the components, or paper over it with one do-everything tool? |
| **Failure handling** | **25** | **Did you catch the data traps? Did you avoid swallowing failures?** |
| Readability & reproducibility | 20 | Is it readable? Same result on a re-run? Is a trace preserved? |

### 꼭 기억할 것 · Remember this

> **못 푼 것보다 틀린 것을 맞다고 하는 것이 더 나쁩니다.**
>
> 답을 못 냈으면 "판단할 수 없다"고 명확히 보고하십시오. (실패 처리 10점)
> 조용히 틀린 값을 확신에 차서 보고하면 5점입니다.

> **EN — presenting something wrong as right is worse than not solving it.**
> If you could not get an answer, report clearly that you cannot judge it
> (10 marks under failure handling). Confidently reporting a quietly wrong number
> scores 5.

### 가산점 (최대 +5) · Bonus marks (up to +5)

- 정리 전후 결과를 **비교해서** 보여줌
- 같은 질문을 여러 번 돌려 **결과의 변동**을 보고
- 유의하지 않은 결과를 **"차이가 있다고 하기 어렵다"** 로 표현
- 표본 크기 불균형이나 검정력 한계를 언급

> **EN**
> - **Comparing** the results before and against after cleaning
> - Running the same question repeatedly and reporting **the variation**
> - Phrasing a non-significant result as **"it is hard to claim a difference"**
> - Noting sample-size imbalance or the limits of statistical power